# Khmer Qwen: Colab improvement experiment
Upload this notebook to Google Colab. Select **Runtime → Change runtime type → GPU**, then **Run all**. Approve the Google Drive mount when prompted.

Target: **CER <20% and ICU Khmer WER <20%** on genuine held-out evaluation. The completed student baseline scored **12.49% CER / 28.81% WER** on 124 validation clips. This new recipe is an experiment, not a promise of reaching 20%.

This run starts from the pinned publisher Khmer Qwen base, with 1,663 official training clips, rank-16 LoRA on q/k/v/o attention projections, learning rate 5e-5, at most four epochs with early stopping after two epochs without improvement, and the same 124 validation clips. It tests a higher learning rate after the previous rank-16 run reached 28.92% validation WER. This is a new experiment from the pinned base. It does not continue the older 883-clip adapter. Test audio is never loaded.

Training runs only in Colab. Checkpoints, predictions and logs are saved to Drive. T4 uses FP32 for compatibility; L4/A100 use BF16. Resume requires the same GPU type and package/source configuration. Setup downloads the model and training data and may take several minutes.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib
if not Path('/content').is_dir():
    raise RuntimeError('Run this notebook in Google Colab.')
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/khmer_qwen_colab')
WORK.mkdir(exist_ok=True)
ENV = Path('/content/khmer_qwen_env')
if not (ENV/'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(ENV)], check=True)
PYTHON = str(ENV/'bin/python')
def run_setup(command):
    print('Running:', ' '.join(command), flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f'Setup failed (exit {code}); see the full output above.')

run_setup([PYTHON, '--version'])
run_setup([sys.executable, '-m', 'pip', '--python', PYTHON,
           'install', '--upgrade', 'pip'])
run_setup([PYTHON, '-m', 'pip', 'install', 'torch==2.8.0', 'torchaudio==2.8.0',
    '--index-url', 'https://download.pytorch.org/whl/cu128'])
run_setup([PYTHON, '-m', 'pip', 'install', 'torch==2.8.0', 'torchaudio==2.8.0',
    'qwen-asr==0.0.6', 'transformers==4.57.6', 'peft==0.21.0', 'accelerate==1.12.0',
    'datasets==5.0.1', 'pyicu-wheels==2.15.2', 'jiwer==4.0.0'])


In [ ]:
PAYLOAD = {'src/train_qwen_candidate.py': '"""Train a reproducible student Qwen LoRA candidate, using validation only.\n\nTargets and masking follow QwenLM/Qwen3-ASR\'s Apache-2.0 fine-tuning example:\nhttps://github.com/QwenLM/Qwen3-ASR/blob/7c6daf77a2421100f5fb066495372c00129d39ff/finetuning/qwen3_asr_sft.py\nCheckpoints retain optimizer, scheduler, and RNG state for continuation.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport math\nimport random\nimport time\nimport types\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport soundfile as sf\nimport torch\nfrom datasets import load_dataset\nfrom peft import LoraConfig, get_peft_model\nfrom qwen_asr import Qwen3ASRModel\nfrom transformers import get_cosine_schedule_with_warmup\n\nfrom matched_fleurs import load_manifest, normalize_text\nfrom qwen_experiment_utils import score_rows\nfrom qwen_khmer_inference import decode_batch\n\nMODEL_ID = \'seanghay/Qwen3-ASR-0.6B-Khmer\'\nREVISION = \'351f1cc594f084e4899ef6cf7bbb17ec64484080\'\n\n\ndef atomic_json(path, value):\n    temp = path.with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    temp.replace(path)\n\n\ndef load_rows(split, indices, cache):\n    ds = load_dataset(\'google/fleurs\', \'km_kh\', split=split)\n    audio = ds.data.column(\'audio\')\n    references = ds.data.column(\'transcription\')\n    cache.mkdir(parents=True, exist_ok=True)\n    rows = []\n    for index in indices:\n        path = cache / f\'{split}_{index}.wav\'\n        if not path.exists():\n            wav, sr = sf.read(io.BytesIO(audio[index].as_py()[\'bytes\']), dtype=\'float32\')\n            if wav.ndim > 1:\n                wav = wav.mean(axis=1)\n            if sr != 16000 or not np.isfinite(wav).all():\n                raise ValueError(\'Invalid training audio\')\n            sf.write(path, wav, sr, subtype=\'PCM_16\')\n        rows.append({\'index\': index, \'audio\': str(path), \'reference\': normalize_text(references[index].as_py())})\n    return rows\n\n\ndef evaluate(wrapper, rows, output, batch_size, beams, force_khmer):\n    wrapper.model.eval()\n    result = {\'purpose\': \'student candidate selection; validation only\',\n              \'split\': \'google/fleurs km_kh validation\', \'test_used\': False,\n              \'decoding\': {\'beams\': beams, \'force_khmer\': force_khmer, \'greedy_retry_on_missing_eos\': True}, \'examples\': []}\n    for start in range(0, len(rows), batch_size):\n        batch = rows[start:start + batch_size]\n        wavs = [sf.read(r[\'audio\'], dtype=\'float32\')[0] for r in batch]\n        predictions = decode_batch(wrapper, wavs, beams=beams, force_khmer=force_khmer)\n        result[\'examples\'].extend({\'validation_index\': r[\'index\'], \'reference\': r[\'reference\'],\n                                   **p} for r, p in zip(batch, predictions))\n        atomic_json(output, result)\n        print(f\'Validation {len(result["examples"])}/{len(rows)}\', flush=True)\n    result[\'metrics\'] = score_rows(result[\'examples\'])\n    atomic_json(output, result)\n    return result[\'metrics\']\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--manifest\', type=Path, default=Path(\'results/diagnostic/qwen_train30_split_20261007.json\'))\n    ap.add_argument(\'--output-dir\', type=Path, required=True)\n    ap.add_argument(\'--cache-dir\', type=Path, default=Path(\'local_eval/qwen_fleurs_audio_expanded\'))\n    ap.add_argument(\'--epochs\', type=int, default=2)\n    ap.add_argument(\'--patience\', type=int, default=0, help=\'Stop after this many epochs without a lower validation WER; zero disables\')\n    ap.add_argument(\'--lr\', type=float, default=1e-5)\n    ap.add_argument(\'--grad-acc\', type=int, default=8)\n    ap.add_argument(\'--rank\', type=int, default=8)\n    ap.add_argument(\'--targets\', default=\'q_proj,v_proj\',\n                    help=\'Comma-separated linear projection names for LoRA\')\n    ap.add_argument(\'--eval-batch-size\', type=int, default=4)\n    ap.add_argument(\'--dtype\', choices=[\'auto\', \'bfloat16\', \'float32\'], default=\'auto\',\n                    help=\'Auto uses BF16 on supported GPUs and FP32 on Colab T4\')\n    ap.add_argument(\'--eval-beams\', type=int, default=1)\n    ap.add_argument(\'--eval-force-khmer\', action=\'store_true\')\n    ap.add_argument(\'--resume\', type=Path)\n    ap.add_argument(\'--max-steps\', type=int, default=0, help=\'Stop after saving a training smoke check; no validation selection\')\n    args = ap.parse_args()\n    if args.patience < 0:\n        ap.error(\'Patience cannot be negative\')\n    if min(args.epochs, args.grad_acc, args.rank, args.eval_batch_size, args.eval_beams) < 1 or args.lr <= 0:\n        ap.error(\'Training and decoding settings must be positive\')\n    targets = [name.strip() for name in args.targets.split(\',\') if name.strip()]\n    if not targets:\n        ap.error(\'At least one LoRA target is required\')\n    torch.set_num_threads(4)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime in Colab\')\n    native_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)\n    dtype_name = (\'bfloat16\' if native_bf16 else \'float32\') if args.dtype == \'auto\' else args.dtype\n    if dtype_name == \'bfloat16\' and not native_bf16:\n        raise RuntimeError(\'This GPU does not support BF16; use --dtype float32\')\n    dtype = getattr(torch, dtype_name)\n    manifest = load_manifest(args.manifest)\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    source_names = [\'train_qwen_candidate.py\', \'qwen_khmer_inference.py\', \'qwen_experiment_utils.py\',\n                    \'score_qwen_rows.py\', \'score_matched_cer_wer.py\', \'matched_fleurs.py\']\n    source_bytes = {name: Path(__file__).with_name(name).read_bytes() for name in source_names}\n    random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)\n    config = {\'base_model\': MODEL_ID, \'base_revision\': REVISION, \'manifest\': str(args.manifest),\n              \'manifest_sha256\': hashlib.sha256(args.manifest.read_bytes()).hexdigest(),\n              \'train_examples\': len(manifest[\'indices\'][\'train\']), \'validation_examples\': len(manifest[\'indices\'][\'validation\']),\n              \'test_used\': False, \'epochs\': args.epochs, \'patience\': args.patience, \'learning_rate\': args.lr,\n              \'gradient_accumulation\': args.grad_acc, \'dtype\': dtype_name, \'seed\': 42,\n              \'lora\': {\'r\': args.rank, \'alpha\': args.rank*2, \'dropout\': .05, \'targets\': targets},\n              \'schedule\': \'cosine, 5% warmup\', \'weight_decay\': .01,\n              \'selection\': \'minimum full validation ICU 77.1 Khmer WER\',\n              \'max_new_tokens\': 512, \'hardware\': torch.cuda.get_device_name(0),\n              \'validation_decoding\': {\'beams\': args.eval_beams, \'force_khmer\': args.eval_force_khmer,\n                                     \'greedy_retry_on_missing_eos\': True},\n              \'decoding_source_sha256\': hashlib.sha256(Path(__file__).with_name(\'qwen_khmer_inference.py\').read_bytes()).hexdigest(),\n              \'source_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n              \'source_files_sha256\': {name: hashlib.sha256(data).hexdigest() for name, data in source_bytes.items()},\n              \'versions\': {name: version(name) for name in (\'torch\',\'transformers\',\'peft\',\'qwen-asr\',\'datasets\')}}\n    config_path = args.output_dir / \'run_config.json\'\n    if config_path.exists():\n        if json.loads(config_path.read_text()) != config:\n            raise ValueError(\'Existing output belongs to a different experiment configuration\')\n        if not args.resume and list(args.output_dir.glob(\'*/training_state.pt\')):\n            raise ValueError(\'Existing output requires matching configuration and explicit --resume\')\n    else:\n        atomic_json(config_path, config)\n    snapshot = args.output_dir / \'source\'\n    snapshot.mkdir(exist_ok=True)\n    for name, data in source_bytes.items():\n        saved = snapshot / name\n        if saved.exists() and saved.read_bytes() != data:\n            raise ValueError(f\'Saved source differs: {name}\')\n        if not saved.exists():\n            saved.write_bytes(data)\n    train = load_rows(\'train\', manifest[\'indices\'][\'train\'], args.cache_dir)\n    validation = load_rows(\'validation\', manifest[\'indices\'][\'validation\'], args.cache_dir)\n    wrapper = Qwen3ASRModel.from_pretrained(MODEL_ID, revision=REVISION, dtype=dtype,\n                device_map=\'cuda:0\', max_inference_batch_size=args.eval_batch_size, max_new_tokens=512)\n    base = wrapper.model\n    def forward(self, **kwargs):\n        return self.thinker(**kwargs)\n    base.forward = types.MethodType(forward, base)\n    model = get_peft_model(base, LoraConfig(r=args.rank, lora_alpha=args.rank*2, lora_dropout=.05,\n                target_modules=targets, bias=\'none\'))\n    # Configure the actual decoder rather than the outer class, which has no\n    # get_input_embeddings implementation. Non-reentrant checkpointing works\n    # with frozen embeddings and trainable LoRA layers.\n    base.thinker.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    base.thinker.audio_tower.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n    wrapper.model = model\n    params = [p for p in model.parameters() if p.requires_grad]\n    print(f\'Trainable parameters: {sum(p.numel() for p in params):,}\', flush=True)\n    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=.01)\n    total_steps = math.ceil(len(train)/args.grad_acc)*args.epochs\n    scheduler = get_cosine_schedule_with_warmup(optimizer, math.ceil(total_steps*.05), total_steps)\n    prefix = wrapper._build_text_prompt(\'\', None)\n    eos = wrapper.processor.tokenizer.eos_token\n    history = []\n    start_epoch, start_position, step = 1, 0, 0\n    resume_order = None\n    resume_loss, resume_count = 0., 0\n    if args.resume:\n        from peft import set_peft_model_state_dict\n        from safetensors.torch import load_file\n        set_peft_model_state_dict(model, load_file(str(args.resume / \'adapter_model.safetensors\')))\n        state = torch.load(args.resume / \'training_state.pt\', map_location=\'cpu\', weights_only=False)\n        if state[\'run_config\'] != config:\n            raise ValueError(\'Resume checkpoint does not match this experiment configuration\')\n        optimizer.load_state_dict(state[\'optimizer\']); scheduler.load_state_dict(state[\'scheduler\'])\n        random.setstate(state[\'python_rng\']); np.random.set_state(state[\'numpy_rng\'])\n        torch.set_rng_state(state[\'torch_rng\']); torch.cuda.set_rng_state_all(state[\'cuda_rng\'])\n        start_epoch, start_position, step = state[\'epoch\'], state[\'position\'], state[\'step\']\n        resume_order = state[\'order\']; history = state[\'history\']\n        resume_loss, resume_count = state[\'epoch_loss_sum\'], state[\'epoch_loss_count\']\n    if (args.output_dir / \'finished.json\').exists():\n        print(\'Experiment already finished; use a new output directory for a new recipe.\', flush=True)\n        return\n    def checkpoint(path, epoch, position, order, loss_sum, loss_count):\n        path.mkdir(parents=True, exist_ok=True)\n        marker = path / \'complete.json\'\n        marker.unlink(missing_ok=True)\n        model.save_pretrained(path)\n        state = {\'run_config\': config, \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(),\n                 \'python_rng\': random.getstate(), \'numpy_rng\': np.random.get_state(),\n                 \'torch_rng\': torch.get_rng_state(), \'cuda_rng\': torch.cuda.get_rng_state_all(),\n                 \'epoch\': epoch, \'position\': position, \'step\': step, \'order\': order, \'history\': history,\n                 \'epoch_loss_sum\': loss_sum, \'epoch_loss_count\': loss_count}\n        torch.save(state, path / \'training_state.pt\')\n        atomic_json(marker, {\'step\': step, \'epoch\': epoch, \'position\': position})\n        print(f\'Saved {path}\', flush=True)\n    t0 = time.perf_counter()\n    for epoch in range(start_epoch, args.epochs+1):\n        order = resume_order if resume_order is not None else random.sample(range(len(train)), len(train))\n        resume_order = None\n        model.train(); optimizer.zero_grad(set_to_none=True)\n        running_loss, count = resume_loss, resume_count\n        resume_loss, resume_count = 0., 0\n        for position in range(start_position, len(order)):\n            row = train[order[position]]\n            wav, sr = sf.read(row[\'audio\'], dtype=\'float32\')\n            target = \'language Khmer<asr_text>\' + row[\'reference\']\n            full = wrapper.processor(text=[prefix+target+eos], audio=[wav], return_tensors=\'pt\', padding=True)\n            prompt = wrapper.processor(text=[prefix], audio=[wav], return_tensors=\'pt\', padding=True)\n            labels = full[\'input_ids\'].clone()\n            labels[:, :int(prompt[\'attention_mask\'].sum())] = -100\n            labels[full[\'attention_mask\'] == 0] = -100\n            full[\'labels\'] = labels\n            full = full.to(\'cuda\').to(dtype)\n            # Confirm BatchFeature.to(dtype) preserved integer IDs and labels.\n            if full[\'labels\'].dtype != torch.int64:\n                raise RuntimeError(\'Labels must remain integer token IDs\')\n            output = model(**full, use_cache=False)\n            loss = output.loss\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f\'Non-finite loss at row {row["index"]}\')\n            running_loss += float(loss.detach()); count += 1\n            group_start = position // args.grad_acc * args.grad_acc\n            group_size = min(args.grad_acc, len(order)-group_start)\n            (loss/group_size).backward()\n            del full, output, loss, prompt, labels\n            if (position+1) % args.grad_acc == 0 or position+1 == len(order):\n                gradient_norm = torch.nn.utils.clip_grad_norm_(params, 1., error_if_nonfinite=True)\n                if gradient_norm <= 0:\n                    raise RuntimeError(\'No trainable gradient reached the adapter\')\n                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True); step += 1\n                if step % 5 == 0 or step == 1:\n                    print(f\'Epoch {epoch}/{args.epochs} step {step}/{total_steps} clips {position+1}/{len(order)} \'\n                          f\'loss {running_loss/count:.4f} lr {scheduler.get_last_lr()[0]:.7f} \'\n                          f\'grad {float(gradient_norm):.4f} elapsed {time.perf_counter()-t0:.0f}s \'\n                          f\'GPUpeak {torch.cuda.max_memory_allocated()/1e9:.2f}GB\', flush=True)\n                if step % 50 == 0 or (args.max_steps and step >= args.max_steps):\n                    checkpoint(args.output_dir / f\'step-{step}\', epoch, position+1, order, running_loss, count)\n                if args.max_steps and step >= args.max_steps:\n                    return\n        start_position = 0\n        path = args.output_dir / f\'epoch-{epoch}\'\n        checkpoint(path, epoch, len(order), order, running_loss, count)\n        torch.cuda.empty_cache()\n        metrics = evaluate(wrapper, validation, path / \'validation_predictions.json\', args.eval_batch_size,\n                           args.eval_beams, args.eval_force_khmer)\n        history.append({\'epoch\': epoch, \'optimizer_steps\': step, \'mean_train_loss\': running_loss/max(count,1), **metrics})\n        atomic_json(args.output_dir / \'history.json\', history)\n        best = min(history, key=lambda r:r[\'wer_percent_icu_segmented\'])\n        atomic_json(args.output_dir / \'best_validation.json\', best)\n        checkpoint(path, epoch+1, 0, None, 0., 0)\n        print(f\'Epoch {epoch}: CER {metrics["cer_percent"]:.2f}%, WER {metrics["wer_percent_icu_segmented"]:.2f}%\', flush=True)\n        if metrics[\'cer_percent\'] < 20 and metrics[\'wer_percent_icu_segmented\'] < 20:\n            atomic_json(args.output_dir / \'finished.json\', {\'reason\': \'validation_target\', \'epoch\': epoch})\n            print(\'Validation thresholds met. Hold out test pending fixed checkpoint selection.\', flush=True)\n            break\n        if args.patience and epoch - best[\'epoch\'] >= args.patience:\n            atomic_json(args.output_dir / \'finished.json\', {\'reason\': \'early_stopping\', \'epoch\': epoch})\n            print(\'Stopping: validation WER did not improve. Keeping the best epoch.\', flush=True)\n            break\n    else:\n        atomic_json(args.output_dir / \'finished.json\', {\'reason\': \'epoch_limit\', \'epoch\': args.epochs})\n\n\nif __name__ == \'__main__\':\n    main()\n', 'src/qwen_khmer_inference.py': '"""Khmer decoding for the publisher\'s fine-tune and its student adapters."""\nimport numpy as np\nimport torch\nfrom qwen_asr.inference.utils import parse_asr_output, split_audio_into_chunks\n\n\ndef decode_batch(wrapper, waveforms, beams=3, force_khmer=True, fallback=True, max_new_tokens=512):\n    """Decode audio, retrying unfinished beam outputs individually with greedy search."""\n    prompt = wrapper._build_text_prompt(\'\', \'Khmer\' if force_khmer else None)\n    inputs = wrapper.processor(text=[prompt]*len(waveforms), audio=waveforms, return_tensors=\'pt\', padding=True)\n    inputs = inputs.to(wrapper.model.device).to(wrapper.model.dtype)\n    with torch.inference_mode():\n        output = wrapper.model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=beams, do_sample=False)\n    tokens = output.sequences[:, inputs[\'input_ids\'].shape[1]:]\n    decoded = wrapper.processor.batch_decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)\n    rows = []\n    for i, (text, ids) in enumerate(zip(decoded, tokens)):\n        has_eos = bool(torch.isin(ids, ids.new_tensor([151645, 151643])).any())\n        row = {\'hypothesis\': parse_asr_output(text, user_language=\'Khmer\' if force_khmer else None)[1],\n               \'has_eos\': has_eos, \'primary_has_eos\': has_eos, \'greedy_fallback\': False,\n               \'generated_tokens_including_padding\': int(ids.numel())}\n        if fallback and beams > 1 and not has_eos:\n            retry = decode_batch(wrapper, [waveforms[i]], beams=1, force_khmer=force_khmer,\n                                 fallback=False, max_new_tokens=max_new_tokens)[0]\n            row.update(retry, primary_has_eos=False, greedy_fallback=True,\n                       primary_hypothesis=row[\'hypothesis\'])\n        rows.append(row)\n    return rows\n\n\ndef transcribe_khmer(wrapper, waveform, sample_rate=16000, beams=3):\n    """Use explicit Khmer decoding and bounded chunks on the laptop GPU.\n\n    The upstream language validator omits Khmer, so build the fine-tune\'s\n    supported language prefix directly. No reference text is supplied.\n    """\n    wav = np.ascontiguousarray(waveform, dtype=np.float32)\n    if sample_rate != 16000 or wav.ndim != 1 or not wav.size or not np.isfinite(wav).all():\n        raise ValueError(\'Qwen needs nonempty, finite, mono 16 kHz audio\')\n    chunks = split_audio_into_chunks(wav=wav, sr=sample_rate, max_chunk_sec=30)\n    texts = []\n    for chunk, _ in chunks:\n        texts.append(decode_batch(wrapper, [chunk], beams=beams)[0][\'hypothesis\'])\n    return \' \'.join(texts).strip()\n', 'src/qwen_experiment_utils.py': '"""Shared experiment utilities, including the original Windows ICU scorer in WSL."""\nimport json\nimport os\nimport subprocess\nfrom pathlib import Path\n\n\ndef score_rows(rows):\n    interpreter = os.environ.get(\'KHMER_SCORE_PYTHON\')\n    if not interpreter:\n        from score_qwen_rows import score_rows as local_score\n        return local_score(rows)\n    # WSL can run the existing Windows scorer without importing native Windows\n    # training libraries. This preserves ICU 77.1 exactly across runtimes.\n    script = str(Path(__file__).with_name(\'score_qwen_rows.py\').resolve())\n    if os.name != \'nt\':\n        script = subprocess.check_output([\'wslpath\', \'-w\', script], text=True).strip()\n    completed = subprocess.run([interpreter, script], input=json.dumps(rows),\n                               text=True, capture_output=True, check=True, timeout=120)\n    return json.loads(completed.stdout)\n', 'src/score_qwen_rows.py': '"""Score JSON reference/hypothesis rows from stdin using the fixed ICU policy."""\nimport json\nimport sys\n\nimport icu\nfrom jiwer import process_characters, process_words\n\nfrom score_matched_cer_wer import cer_text, word_tokens\n\n\ndef score_rows(rows):\n    if icu.ICU_VERSION != \'77.1\':\n        raise RuntimeError(f\'Experiments require ICU 77.1; found {icu.ICU_VERSION}\')\n    chars = process_characters([cer_text(r[\'reference\']) for r in rows],\n                               [cer_text(r[\'hypothesis\']) for r in rows])\n    words = process_words([\' \'.join(word_tokens(r[\'reference\'])) for r in rows],\n                          [\' \'.join(word_tokens(r[\'hypothesis\'])) for r in rows])\n    return {\'evaluated_examples\': len(rows), \'cer_percent\': 100 * chars.cer,\n            \'wer_percent_icu_segmented\': 100 * words.wer,\n            \'cer_reference_characters\': chars.hits + chars.substitutions + chars.deletions,\n            \'wer_reference_words\': words.hits + words.substitutions + words.deletions,\n            \'cer_errors\': chars.substitutions + chars.deletions + chars.insertions,\n            \'wer_errors\': words.substitutions + words.deletions + words.insertions,\n            \'icu_version\': icu.ICU_VERSION}\n\n\nif __name__ == \'__main__\':\n    print(json.dumps(score_rows(json.load(sys.stdin))))\n', 'src/score_matched_cer_wer.py': '"""Score saved FLEURS predictions with matched CER and Khmer-segmented WER.\n\nWER uses ICU\'s Khmer word-break dictionary on both references and hypotheses.\nIt is a post-hoc metric: no model outputs or reference text are changed.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport re\nimport unicodedata\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport icu\nfrom jiwer import process_characters, process_words\n\n\ndef clean(text: str) -> str:\n    return unicodedata.normalize("NFC", text).replace("\\u200b", "").replace("\\ufeff", "")\n\n\ndef cer_text(text: str) -> str:\n    return re.sub(r"\\s+", "", clean(text))\n\n\ndef word_tokens(text: str) -> list[str]:\n    value = icu.UnicodeString(clean(text))\n    breaker = icu.BreakIterator.createWordInstance(icu.Locale("km_KH"))\n    breaker.setText(value)\n    start = breaker.first()\n    tokens = []\n    for end in breaker:\n        # ICU status 0 marks punctuation and whitespace. Word, number, and\n        # letter segments have positive statuses and count as words here.\n        if breaker.getRuleStatus() > 0:\n            tokens.append(str(value[start:end]))\n        start = end\n    return tokens\n\n\ndef load_pairs(path: Path) -> tuple[list[int], list[str], list[str], dict]:\n    data = json.loads(path.read_text(encoding="utf-8"))\n    if "examples" in data:\n        rows = data["examples"]\n        indices = [row["test_index"] for row in rows]\n        references = [row["reference"] for row in rows]\n        hypotheses = [row["hypothesis"] for row in rows]\n    else:\n        indices = data["test_indices"]\n        references = data["references"]\n        hypotheses = data["hypotheses"]\n    if not isinstance(indices, list) or not (len(indices) == len(references) == len(hypotheses)):\n        raise ValueError(f"Invalid or incomplete prediction pairs: {path}")\n    if len(set(indices)) != len(indices):\n        raise ValueError(f"Duplicate test indices in {path}")\n    return indices, references, hypotheses, data\n\n\ndef score(path: Path, expected_indices: list[int]) -> tuple[dict, list[str]]:\n    indices, references, hypotheses, data = load_pairs(path)\n    if indices != expected_indices:\n        raise ValueError(f"Prediction indices do not match the saved test manifest: {path}")\n    normalized_references = [cer_text(text) for text in references]\n    normalized_hypotheses = [cer_text(text) for text in hypotheses]\n    if any(not text for text in normalized_references):\n        raise ValueError(f"Empty normalized reference in {path}")\n\n    characters = process_characters(normalized_references, normalized_hypotheses)\n    reference_words = [word_tokens(text) for text in references]\n    hypothesis_words = [word_tokens(text) for text in hypotheses]\n    if any(not words for words in reference_words):\n        raise ValueError(f"Empty word segmentation of a reference in {path}")\n    words = process_words(\n        [" ".join(tokens) for tokens in reference_words],\n        [" ".join(tokens) for tokens in hypothesis_words],\n    )\n    saved_cer = data.get("cer_percent_whitespace_removed", data.get("test_cer"))\n    if saved_cer is not None and abs(100 * characters.cer - float(saved_cer)) > 0.01:\n        raise ValueError(f"Recomputed CER disagrees with saved result in {path}")\n    return {\n        "prediction_file": path.as_posix(),\n        "test_examples": len(indices),\n        "cer_percent": 100 * characters.cer,\n        "cer_reference_characters": characters.hits + characters.substitutions + characters.deletions,\n        "cer_substitutions": characters.substitutions,\n        "cer_deletions": characters.deletions,\n        "cer_insertions": characters.insertions,\n        "wer_percent_icu_segmented": 100 * words.wer,\n        "wer_reference_words": words.hits + words.substitutions + words.deletions,\n        "wer_substitutions": words.substitutions,\n        "wer_deletions": words.deletions,\n        "wer_insertions": words.insertions,\n    }, normalized_references\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--manifest", type=Path, default=Path("results/matched_fleurs_split.json"))\n    parser.add_argument("--whisper", type=Path)\n    parser.add_argument("--mms", type=Path)\n    parser.add_argument("--frozen-mms", type=Path, help="Optional third approach: frozen MMS encoder with trained CTC head.")\n    parser.add_argument("--output", type=Path, default=Path("results/matched_cer_wer_summary.json"))\n    args = parser.parse_args()\n    if not args.whisper and not args.mms and not args.frozen_mms:\n        parser.error("Provide at least one saved prediction file")\n    manifest = json.loads(args.manifest.read_text(encoding="utf-8"))\n    expected_indices = manifest["indices"]["test"]\n    models = {}\n    shared_references = None\n    for name, path in (("whisper_tiny", args.whisper), ("mms_1b_ctc", args.mms), ("mms_frozen_ctc", args.frozen_mms)):\n        if path is None:\n            continue\n        models[name], references = score(path, expected_indices)\n        if shared_references is not None and references != shared_references:\n            raise ValueError("The models\' normalized test references differ")\n        shared_references = references\n    result = {\n        "dataset": "google/fleurs km_kh held-out test",\n        "manifest": args.manifest.as_posix(),\n        "test_examples": len(expected_indices),\n        "cer_policy": "NFC; remove U+200B/U+FEFF and all whitespace; punctuation retained",\n        "wer_policy": "NFC; remove U+200B/U+FEFF; ICU km_KH word breaks; discard punctuation and whitespace segments; corpus edit distance / reference words",\n        "icu_version": icu.ICU_VERSION,\n        "pyicu_wheels_version": version("pyicu-wheels"),\n        "models": models,\n        "comparison_complete": "whisper_tiny" in models and "mms_1b_ctc" in models,\n        "third_approach_complete": "mms_frozen_ctc" in models,\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(result, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    for name, metrics in models.items():\n        print(f"{name}: CER {metrics[\'cer_percent\']:.2f}%, ICU WER {metrics[\'wer_percent_icu_segmented\']:.2f}%")\n    print(f"Saved {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/matched_fleurs.py': '"""Create and load one fixed FLEURS row manifest for both ASR approaches.\n\nRun ``python src/matched_fleurs.py`` before the matched Colab experiments.\nRows are selected from the official FLEURS splits, then the same eligibility\nrules are applied before either model sees a row.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport io\nimport json\nimport random\nimport re\nimport unicodedata\nfrom pathlib import Path\n\nimport soundfile as sf\n\nDATASET = "google/fleurs"\nCONFIG = "km_kh"\nSPLITS = ("train", "validation", "test")\n\n\ndef load_manifest(path: str | Path) -> dict:\n    manifest = json.loads(Path(path).read_text(encoding="utf-8"))\n    if manifest.get("schema_version") != 1 or manifest.get("dataset") != DATASET or manifest.get("config") != CONFIG:\n        raise ValueError("Expected a version-1 Google FLEURS km_kh split manifest")\n    for split in SPLITS:\n        rows = manifest.get("indices", {}).get(split)\n        if not isinstance(rows, list) or not rows or any(type(i) is not int or i < 0 for i in rows):\n            raise ValueError(f"Invalid or empty {split} indices in split manifest")\n        if len(rows) != len(set(rows)):\n            raise ValueError(f"Duplicate {split} indices in split manifest")\n    return manifest\n\n\ndef select_manifest_split(dataset, manifest: dict, split: str):\n    indices = manifest["indices"][split]\n    if max(indices) >= len(dataset):\n        raise ValueError(f"{split} manifest index exceeds the loaded FLEURS split")\n    return dataset.select(indices)\n\n\ndef normalize_text(value: str) -> str:\n    text = unicodedata.normalize("NFC", str(value or ""))\n    text = text.replace("\\u200b", "").replace("\\ufeff", "")\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, default=Path("results/matched_fleurs_split.json"))\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--train-candidates", type=int, default=1000)\n    parser.add_argument("--validation-candidates", type=int, default=200)\n    parser.add_argument("--test-candidates", type=int, default=200)\n    parser.add_argument("--test-start", type=int, default=200, help="Skip earlier test rows already inspected in historical runs.")\n    parser.add_argument("--max-duration-seconds", type=float, default=15.0)\n    parser.add_argument("--whisper-max-label-tokens", type=int, default=448)\n    args = parser.parse_args()\n\n    from datasets import load_dataset\n    from transformers import WhisperTokenizer\n\n    fleurs = load_dataset(DATASET, CONFIG)\n    tokenizer = WhisperTokenizer.from_pretrained(\n        "openai/whisper-tiny", language="Khmer", task="transcribe"\n    )\n    candidate_limits = {\n        "train": args.train_candidates,\n        "validation": args.validation_candidates,\n        "test": args.test_candidates,\n    }\n    indices = {}\n    for split in SPLITS:\n        dataset = fleurs[split]\n        audio_column = dataset.data.column("audio")\n        text_column = dataset.data.column("transcription")\n        candidates = list(range(len(dataset)))\n        if split == "train":\n            random.Random(args.seed).shuffle(candidates)\n        elif split == "test":\n            candidates = candidates[args.test_start:]\n        candidates = candidates[: candidate_limits[split]]\n        kept = []\n        for index in candidates:\n            record = audio_column[index].as_py()\n            if record["bytes"] is None:\n                raise ValueError(f"FLEURS {split} row {index} has no embedded audio bytes")\n            with sf.SoundFile(io.BytesIO(record["bytes"])) as audio_file:\n                duration = len(audio_file) / audio_file.samplerate\n            label_length = len(tokenizer(normalize_text(text_column[index].as_py())).input_ids)\n            if duration <= args.max_duration_seconds and label_length < args.whisper_max_label_tokens:\n                kept.append(index)\n        if not kept:\n            raise ValueError(f"No {split} rows remain after shared filtering")\n        indices[split] = kept\n        print(f"{split}: {len(kept)} of {len(candidates)} candidate rows retained")\n\n    manifest = {\n        "schema_version": 1,\n        "dataset": DATASET,\n        "config": CONFIG,\n        "seed": args.seed,\n        "selection": "Seeded Python shuffle for train; official validation/test order",\n        "test_start": args.test_start,\n        "eligibility": {\n            "max_duration_seconds": args.max_duration_seconds,\n            "whisper_max_label_tokens": args.whisper_max_label_tokens,\n        },\n        "indices": indices,\n    }\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(f"Saved shared row manifest: {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'results/diagnostic/qwen_train30_split_20261007.json': '{\n  "schema_version": 1,\n  "dataset": "google/fleurs",\n  "config": "km_kh",\n  "seed": 42,\n  "selection": "All official train clips <=30 seconds with finite, non-empty audio and non-empty transcript; original validation and test indices unchanged",\n  "not_a_matched_training_size_comparison": true,\n  "parent_manifest": "results/diagnostic/qwen_expanded_train_split.json",\n  "eligibility": {\n    "max_duration_seconds": 30.0,\n    "nonempty_audio_and_transcript": true\n  },\n  "indices": {\n    "train": [\n      0,\n      1,\n      2,\n      3,\n      4,\n      5,\n      6,\n      7,\n      8,\n      9,\n      10,\n      11,\n      12,\n      13,\n      14,\n      15,\n      16,\n      17,\n      18,\n      19,\n      20,\n      21,\n      22,\n      23,\n      24,\n      25,\n      26,\n      27,\n      28,\n      29,\n      30,\n      31,\n      32,\n      33,\n      34,\n      35,\n      36,\n      37,\n      38,\n      39,\n      40,\n      41,\n      42,\n      43,\n      44,\n      45,\n      46,\n      47,\n      48,\n      49,\n      50,\n      51,\n      52,\n      53,\n      54,\n      55,\n      56,\n      57,\n      58,\n      59,\n      60,\n      61,\n      62,\n      63,\n      64,\n      65,\n      66,\n      67,\n      68,\n      69,\n      70,\n      71,\n      72,\n      73,\n      74,\n      75,\n      76,\n      77,\n      78,\n      79,\n      80,\n      81,\n      82,\n      83,\n      84,\n      85,\n      86,\n      87,\n      88,\n      89,\n      90,\n      91,\n      92,\n      93,\n      94,\n      95,\n      96,\n      97,\n      98,\n      99,\n      100,\n      101,\n      102,\n      103,\n      104,\n      105,\n      106,\n      107,\n      108,\n      109,\n      110,\n      111,\n      112,\n      113,\n      114,\n      115,\n      116,\n      117,\n      118,\n      119,\n      121,\n      122,\n      123,\n      124,\n      125,\n      126,\n      127,\n      128,\n      129,\n      130,\n      131,\n      132,\n      133,\n      134,\n      135,\n      136,\n      137,\n      138,\n      139,\n      140,\n      141,\n      142,\n      143,\n      144,\n      145,\n      146,\n      147,\n      148,\n      149,\n      150,\n      151,\n      152,\n      153,\n      154,\n      155,\n      156,\n      157,\n      158,\n      159,\n      160,\n      161,\n      162,\n      163,\n      164,\n      165,\n      166,\n      167,\n      168,\n      169,\n      170,\n      171,\n      172,\n      173,\n      174,\n      175,\n      176,\n      177,\n      178,\n      179,\n      180,\n      182,\n      184,\n      185,\n      187,\n      188,\n      189,\n      190,\n      191,\n      192,\n      193,\n      194,\n      195,\n      196,\n      197,\n      198,\n      199,\n      200,\n      201,\n      202,\n      203,\n      204,\n      205,\n      206,\n      207,\n      208,\n      209,\n      210,\n      211,\n      212,\n      213,\n      214,\n      215,\n      216,\n      217,\n      218,\n      219,\n      220,\n      221,\n      222,\n      223,\n      224,\n      225,\n      226,\n      227,\n      228,\n      229,\n      230,\n      231,\n      232,\n      233,\n      234,\n      235,\n      236,\n      237,\n      238,\n      239,\n      240,\n      241,\n      242,\n      243,\n      244,\n      245,\n      246,\n      247,\n      248,\n      249,\n      250,\n      251,\n      252,\n      253,\n      254,\n      255,\n      256,\n      257,\n      258,\n      259,\n      260,\n      261,\n      262,\n      263,\n      264,\n      265,\n      266,\n      267,\n      268,\n      269,\n      270,\n      271,\n      272,\n      273,\n      274,\n      275,\n      276,\n      277,\n      278,\n      279,\n      280,\n      281,\n      282,\n      283,\n      284,\n      285,\n      286,\n      287,\n      288,\n      289,\n      290,\n      291,\n      292,\n      293,\n      294,\n      295,\n      296,\n      297,\n      298,\n      299,\n      300,\n      301,\n      302,\n      303,\n      304,\n      305,\n      306,\n      307,\n      308,\n      309,\n      310,\n      311,\n      312,\n      313,\n      314,\n      315,\n      316,\n      317,\n      318,\n      319,\n      320,\n      321,\n      322,\n      323,\n      324,\n      325,\n      326,\n      327,\n      328,\n      329,\n      330,\n      331,\n      332,\n      333,\n      334,\n      336,\n      337,\n      338,\n      339,\n      340,\n      341,\n      342,\n      344,\n      345,\n      346,\n      347,\n      348,\n      349,\n      350,\n      351,\n      352,\n      353,\n      354,\n      355,\n      356,\n      357,\n      358,\n      359,\n      360,\n      361,\n      362,\n      363,\n      364,\n      365,\n      366,\n      367,\n      368,\n      369,\n      370,\n      371,\n      372,\n      373,\n      374,\n      375,\n      376,\n      377,\n      378,\n      379,\n      380,\n      381,\n      382,\n      383,\n      384,\n      385,\n      386,\n      387,\n      388,\n      389,\n      390,\n      391,\n      392,\n      393,\n      394,\n      395,\n      396,\n      397,\n      398,\n      399,\n      400,\n      401,\n      402,\n      403,\n      404,\n      405,\n      406,\n      407,\n      408,\n      409,\n      410,\n      411,\n      412,\n      413,\n      414,\n      415,\n      416,\n      417,\n      418,\n      419,\n      420,\n      421,\n      422,\n      423,\n      424,\n      425,\n      426,\n      427,\n      428,\n      429,\n      430,\n      431,\n      432,\n      433,\n      434,\n      435,\n      436,\n      437,\n      438,\n      439,\n      440,\n      441,\n      442,\n      443,\n      444,\n      445,\n      446,\n      447,\n      448,\n      449,\n      450,\n      451,\n      452,\n      453,\n      454,\n      455,\n      456,\n      457,\n      458,\n      459,\n      460,\n      461,\n      462,\n      463,\n      464,\n      465,\n      466,\n      467,\n      468,\n      469,\n      470,\n      471,\n      472,\n      473,\n      474,\n      475,\n      476,\n      477,\n      478,\n      479,\n      480,\n      481,\n      482,\n      483,\n      484,\n      485,\n      486,\n      487,\n      488,\n      489,\n      490,\n      491,\n      492,\n      493,\n      494,\n      495,\n      496,\n      497,\n      498,\n      499,\n      500,\n      501,\n      502,\n      503,\n      504,\n      505,\n      506,\n      507,\n      508,\n      509,\n      510,\n      511,\n      512,\n      513,\n      514,\n      515,\n      516,\n      517,\n      518,\n      519,\n      520,\n      521,\n      522,\n      523,\n      524,\n      525,\n      526,\n      527,\n      528,\n      529,\n      530,\n      531,\n      532,\n      533,\n      534,\n      535,\n      536,\n      537,\n      538,\n      539,\n      540,\n      541,\n      542,\n      543,\n      544,\n      545,\n      546,\n      547,\n      548,\n      549,\n      550,\n      551,\n      552,\n      553,\n      554,\n      555,\n      556,\n      557,\n      558,\n      559,\n      560,\n      561,\n      562,\n      563,\n      564,\n      565,\n      566,\n      567,\n      568,\n      569,\n      570,\n      571,\n      572,\n      573,\n      574,\n      575,\n      576,\n      577,\n      578,\n      579,\n      580,\n      581,\n      582,\n      583,\n      584,\n      585,\n      586,\n      587,\n      588,\n      589,\n      590,\n      591,\n      592,\n      593,\n      594,\n      595,\n      596,\n      597,\n      598,\n      599,\n      600,\n      601,\n      602,\n      603,\n      604,\n      605,\n      606,\n      607,\n      608,\n      609,\n      610,\n      611,\n      612,\n      613,\n      614,\n      615,\n      616,\n      617,\n      618,\n      619,\n      620,\n      621,\n      622,\n      623,\n      624,\n      625,\n      626,\n      627,\n      628,\n      629,\n      630,\n      631,\n      632,\n      633,\n      634,\n      635,\n      636,\n      637,\n      638,\n      639,\n      640,\n      641,\n      642,\n      643,\n      644,\n      645,\n      646,\n      647,\n      648,\n      649,\n      650,\n      651,\n      652,\n      653,\n      654,\n      655,\n      656,\n      657,\n      658,\n      659,\n      660,\n      661,\n      662,\n      663,\n      664,\n      665,\n      666,\n      667,\n      668,\n      669,\n      670,\n      671,\n      672,\n      673,\n      674,\n      675,\n      676,\n      677,\n      678,\n      679,\n      680,\n      681,\n      682,\n      683,\n      684,\n      685,\n      686,\n      687,\n      688,\n      689,\n      690,\n      691,\n      692,\n      693,\n      694,\n      695,\n      696,\n      697,\n      698,\n      699,\n      700,\n      701,\n      702,\n      703,\n      704,\n      705,\n      706,\n      707,\n      708,\n      709,\n      710,\n      711,\n      712,\n      713,\n      714,\n      715,\n      716,\n      717,\n      718,\n      719,\n      720,\n      721,\n      722,\n      723,\n      724,\n      725,\n      726,\n      727,\n      728,\n      729,\n      730,\n      731,\n      732,\n      733,\n      734,\n      735,\n      736,\n      737,\n      738,\n      739,\n      740,\n      741,\n      742,\n      743,\n      744,\n      745,\n      746,\n      747,\n      748,\n      749,\n      750,\n      751,\n      752,\n      753,\n      754,\n      755,\n      756,\n      757,\n      759,\n      760,\n      761,\n      762,\n      763,\n      764,\n      765,\n      766,\n      767,\n      768,\n      769,\n      770,\n      771,\n      772,\n      773,\n      774,\n      775,\n      776,\n      777,\n      778,\n      779,\n      780,\n      781,\n      782,\n      783,\n      784,\n      785,\n      786,\n      787,\n      788,\n      789,\n      790,\n      791,\n      792,\n      793,\n      794,\n      795,\n      796,\n      797,\n      798,\n      799,\n      800,\n      801,\n      802,\n      803,\n      804,\n      805,\n      806,\n      807,\n      808,\n      809,\n      810,\n      811,\n      812,\n      813,\n      814,\n      815,\n      816,\n      817,\n      818,\n      819,\n      820,\n      821,\n      822,\n      823,\n      824,\n      825,\n      826,\n      827,\n      828,\n      829,\n      830,\n      831,\n      832,\n      833,\n      834,\n      835,\n      836,\n      837,\n      838,\n      839,\n      840,\n      841,\n      842,\n      843,\n      844,\n      845,\n      846,\n      847,\n      848,\n      849,\n      850,\n      851,\n      852,\n      853,\n      854,\n      855,\n      856,\n      857,\n      858,\n      859,\n      860,\n      861,\n      862,\n      863,\n      864,\n      865,\n      866,\n      867,\n      868,\n      869,\n      870,\n      871,\n      872,\n      873,\n      874,\n      875,\n      877,\n      878,\n      879,\n      880,\n      881,\n      882,\n      883,\n      884,\n      885,\n      886,\n      887,\n      888,\n      889,\n      890,\n      891,\n      892,\n      893,\n      894,\n      895,\n      896,\n      897,\n      898,\n      899,\n      900,\n      901,\n      902,\n      903,\n      904,\n      905,\n      906,\n      907,\n      908,\n      909,\n      910,\n      911,\n      912,\n      913,\n      914,\n      915,\n      916,\n      917,\n      918,\n      919,\n      920,\n      921,\n      922,\n      923,\n      924,\n      925,\n      926,\n      927,\n      928,\n      929,\n      930,\n      931,\n      932,\n      933,\n      934,\n      936,\n      937,\n      938,\n      939,\n      940,\n      941,\n      942,\n      943,\n      944,\n      945,\n      946,\n      947,\n      948,\n      949,\n      950,\n      951,\n      952,\n      953,\n      954,\n      955,\n      956,\n      957,\n      958,\n      959,\n      960,\n      961,\n      962,\n      963,\n      964,\n      965,\n      966,\n      967,\n      968,\n      969,\n      970,\n      971,\n      972,\n      973,\n      974,\n      975,\n      976,\n      977,\n      978,\n      979,\n      980,\n      981,\n      982,\n      983,\n      984,\n      985,\n      986,\n      987,\n      988,\n      989,\n      990,\n      991,\n      992,\n      993,\n      994,\n      995,\n      996,\n      997,\n      998,\n      999,\n      1000,\n      1001,\n      1002,\n      1003,\n      1004,\n      1005,\n      1006,\n      1007,\n      1008,\n      1009,\n      1010,\n      1011,\n      1012,\n      1013,\n      1014,\n      1015,\n      1016,\n      1017,\n      1018,\n      1019,\n      1020,\n      1021,\n      1022,\n      1023,\n      1024,\n      1025,\n      1026,\n      1027,\n      1028,\n      1029,\n      1030,\n      1031,\n      1032,\n      1033,\n      1034,\n      1035,\n      1036,\n      1037,\n      1038,\n      1039,\n      1040,\n      1041,\n      1042,\n      1043,\n      1044,\n      1045,\n      1046,\n      1047,\n      1048,\n      1049,\n      1050,\n      1051,\n      1052,\n      1053,\n      1054,\n      1055,\n      1056,\n      1057,\n      1058,\n      1059,\n      1060,\n      1061,\n      1062,\n      1063,\n      1064,\n      1065,\n      1066,\n      1067,\n      1068,\n      1069,\n      1070,\n      1071,\n      1072,\n      1073,\n      1074,\n      1075,\n      1077,\n      1078,\n      1079,\n      1080,\n      1081,\n      1082,\n      1083,\n      1084,\n      1085,\n      1086,\n      1087,\n      1088,\n      1089,\n      1090,\n      1091,\n      1092,\n      1093,\n      1094,\n      1095,\n      1096,\n      1097,\n      1098,\n      1099,\n      1100,\n      1101,\n      1102,\n      1103,\n      1104,\n      1105,\n      1106,\n      1107,\n      1108,\n      1109,\n      1110,\n      1111,\n      1112,\n      1113,\n      1114,\n      1115,\n      1116,\n      1117,\n      1118,\n      1119,\n      1120,\n      1121,\n      1122,\n      1123,\n      1125,\n      1126,\n      1127,\n      1128,\n      1129,\n      1130,\n      1131,\n      1132,\n      1133,\n      1134,\n      1135,\n      1136,\n      1137,\n      1138,\n      1139,\n      1140,\n      1141,\n      1142,\n      1143,\n      1144,\n      1145,\n      1146,\n      1147,\n      1148,\n      1149,\n      1150,\n      1151,\n      1152,\n      1153,\n      1154,\n      1155,\n      1156,\n      1157,\n      1158,\n      1159,\n      1160,\n      1161,\n      1162,\n      1163,\n      1164,\n      1165,\n      1166,\n      1167,\n      1168,\n      1169,\n      1170,\n      1171,\n      1172,\n      1173,\n      1174,\n      1175,\n      1176,\n      1177,\n      1178,\n      1179,\n      1180,\n      1181,\n      1182,\n      1183,\n      1184,\n      1185,\n      1186,\n      1187,\n      1188,\n      1189,\n      1190,\n      1191,\n      1192,\n      1193,\n      1194,\n      1195,\n      1196,\n      1197,\n      1198,\n      1199,\n      1200,\n      1201,\n      1202,\n      1203,\n      1204,\n      1205,\n      1206,\n      1207,\n      1208,\n      1209,\n      1210,\n      1211,\n      1212,\n      1213,\n      1214,\n      1215,\n      1216,\n      1217,\n      1218,\n      1219,\n      1220,\n      1221,\n      1222,\n      1223,\n      1224,\n      1225,\n      1226,\n      1227,\n      1228,\n      1229,\n      1230,\n      1231,\n      1232,\n      1233,\n      1234,\n      1235,\n      1236,\n      1237,\n      1238,\n      1239,\n      1240,\n      1241,\n      1242,\n      1243,\n      1244,\n      1245,\n      1246,\n      1247,\n      1248,\n      1249,\n      1250,\n      1251,\n      1252,\n      1253,\n      1254,\n      1255,\n      1256,\n      1257,\n      1259,\n      1260,\n      1261,\n      1262,\n      1263,\n      1264,\n      1265,\n      1266,\n      1267,\n      1268,\n      1269,\n      1270,\n      1271,\n      1272,\n      1273,\n      1274,\n      1275,\n      1276,\n      1277,\n      1278,\n      1279,\n      1280,\n      1281,\n      1282,\n      1283,\n      1284,\n      1285,\n      1286,\n      1287,\n      1288,\n      1289,\n      1290,\n      1291,\n      1292,\n      1293,\n      1294,\n      1295,\n      1296,\n      1297,\n      1298,\n      1299,\n      1300,\n      1301,\n      1302,\n      1303,\n      1304,\n      1305,\n      1306,\n      1307,\n      1308,\n      1309,\n      1310,\n      1311,\n      1312,\n      1313,\n      1314,\n      1315,\n      1316,\n      1317,\n      1318,\n      1319,\n      1320,\n      1321,\n      1322,\n      1323,\n      1324,\n      1325,\n      1326,\n      1327,\n      1328,\n      1329,\n      1330,\n      1331,\n      1332,\n      1333,\n      1334,\n      1335,\n      1336,\n      1337,\n      1338,\n      1339,\n      1340,\n      1341,\n      1342,\n      1343,\n      1344,\n      1345,\n      1346,\n      1347,\n      1348,\n      1349,\n      1350,\n      1351,\n      1352,\n      1353,\n      1354,\n      1355,\n      1356,\n      1357,\n      1358,\n      1359,\n      1360,\n      1361,\n      1362,\n      1363,\n      1364,\n      1365,\n      1366,\n      1367,\n      1368,\n      1369,\n      1370,\n      1371,\n      1372,\n      1373,\n      1374,\n      1375,\n      1376,\n      1377,\n      1378,\n      1379,\n      1380,\n      1381,\n      1382,\n      1383,\n      1384,\n      1385,\n      1386,\n      1387,\n      1388,\n      1389,\n      1390,\n      1391,\n      1392,\n      1393,\n      1394,\n      1395,\n      1396,\n      1397,\n      1398,\n      1399,\n      1400,\n      1401,\n      1402,\n      1403,\n      1404,\n      1405,\n      1406,\n      1407,\n      1408,\n      1409,\n      1410,\n      1411,\n      1412,\n      1413,\n      1414,\n      1415,\n      1416,\n      1417,\n      1418,\n      1419,\n      1420,\n      1421,\n      1422,\n      1423,\n      1424,\n      1425,\n      1426,\n      1427,\n      1428,\n      1429,\n      1430,\n      1431,\n      1432,\n      1433,\n      1434,\n      1435,\n      1436,\n      1437,\n      1438,\n      1439,\n      1440,\n      1441,\n      1442,\n      1443,\n      1444,\n      1445,\n      1446,\n      1447,\n      1448,\n      1449,\n      1450,\n      1451,\n      1452,\n      1453,\n      1454,\n      1455,\n      1456,\n      1457,\n      1458,\n      1459,\n      1460,\n      1461,\n      1462,\n      1463,\n      1464,\n      1465,\n      1466,\n      1467,\n      1468,\n      1469,\n      1470,\n      1471,\n      1472,\n      1473,\n      1474,\n      1475,\n      1476,\n      1477,\n      1478,\n      1479,\n      1480,\n      1481,\n      1482,\n      1483,\n      1484,\n      1485,\n      1486,\n      1487,\n      1488,\n      1489,\n      1490,\n      1491,\n      1492,\n      1493,\n      1494,\n      1495,\n      1496,\n      1497,\n      1498,\n      1499,\n      1500,\n      1501,\n      1502,\n      1503,\n      1504,\n      1505,\n      1506,\n      1507,\n      1508,\n      1509,\n      1510,\n      1511,\n      1512,\n      1513,\n      1514,\n      1515,\n      1516,\n      1517,\n      1518,\n      1519,\n      1520,\n      1521,\n      1522,\n      1523,\n      1524,\n      1525,\n      1526,\n      1527,\n      1528,\n      1529,\n      1530,\n      1531,\n      1532,\n      1533,\n      1534,\n      1535,\n      1536,\n      1537,\n      1538,\n      1539,\n      1540,\n      1541,\n      1542,\n      1543,\n      1544,\n      1545,\n      1546,\n      1547,\n      1548,\n      1549,\n      1550,\n      1551,\n      1552,\n      1553,\n      1554,\n      1555,\n      1556,\n      1557,\n      1558,\n      1559,\n      1560,\n      1561,\n      1562,\n      1563,\n      1564,\n      1565,\n      1566,\n      1567,\n      1568,\n      1569,\n      1570,\n      1571,\n      1572,\n      1573,\n      1574,\n      1575,\n      1576,\n      1577,\n      1578,\n      1579,\n      1580,\n      1581,\n      1582,\n      1583,\n      1584,\n      1585,\n      1586,\n      1587,\n      1588,\n      1589,\n      1590,\n      1591,\n      1592,\n      1593,\n      1594,\n      1595,\n      1596,\n      1597,\n      1598,\n      1599,\n      1600,\n      1601,\n      1602,\n      1603,\n      1604,\n      1605,\n      1606,\n      1607,\n      1608,\n      1609,\n      1610,\n      1611,\n      1612,\n      1613,\n      1614,\n      1615,\n      1616,\n      1617,\n      1618,\n      1619,\n      1620,\n      1621,\n      1622,\n      1623,\n      1624,\n      1625,\n      1626,\n      1627,\n      1628,\n      1629,\n      1630,\n      1631,\n      1632,\n      1633,\n      1634,\n      1635,\n      1636,\n      1637,\n      1638,\n      1639,\n      1640,\n      1641,\n      1642,\n      1643,\n      1644,\n      1645,\n      1646,\n      1647,\n      1648,\n      1649,\n      1650,\n      1651,\n      1652,\n      1653,\n      1654,\n      1655,\n      1656,\n      1657,\n      1658,\n      1659,\n      1660,\n      1661,\n      1662,\n      1663,\n      1664,\n      1665,\n      1666,\n      1667,\n      1668,\n      1669,\n      1670,\n      1671,\n      1672,\n      1673,\n      1674\n    ],\n    "validation": [\n      0,\n      2,\n      3,\n      7,\n      8,\n      9,\n      10,\n      12,\n      14,\n      16,\n      21,\n      22,\n      24,\n      25,\n      26,\n      28,\n      29,\n      30,\n      31,\n      33,\n      35,\n      36,\n      37,\n      38,\n      39,\n      40,\n      42,\n      43,\n      44,\n      46,\n      47,\n      49,\n      50,\n      51,\n      52,\n      53,\n      54,\n      57,\n      58,\n      59,\n      60,\n      62,\n      63,\n      65,\n      68,\n      71,\n      72,\n      73,\n      74,\n      76,\n      77,\n      78,\n      82,\n      83,\n      84,\n      85,\n      86,\n      87,\n      88,\n      90,\n      91,\n      92,\n      94,\n      95,\n      96,\n      97,\n      98,\n      100,\n      102,\n      103,\n      104,\n      105,\n      106,\n      107,\n      108,\n      110,\n      111,\n      112,\n      113,\n      115,\n      116,\n      119,\n      121,\n      125,\n      126,\n      127,\n      128,\n      130,\n      131,\n      134,\n      135,\n      139,\n      140,\n      141,\n      142,\n      145,\n      147,\n      149,\n      150,\n      153,\n      155,\n      156,\n      158,\n      160,\n      161,\n      162,\n      164,\n      165,\n      166,\n      167,\n      169,\n      171,\n      172,\n      173,\n      174,\n      177,\n      178,\n      180,\n      184,\n      190,\n      194,\n      196,\n      197,\n      198\n    ],\n    "test": [\n      201,\n      202,\n      203,\n      204,\n      206,\n      209,\n      211,\n      215,\n      216,\n      217,\n      218,\n      219,\n      220,\n      221,\n      222,\n      223,\n      224,\n      226,\n      230,\n      231,\n      234,\n      235,\n      236,\n      237,\n      238,\n      239,\n      240,\n      241,\n      243,\n      245,\n      246,\n      249,\n      250,\n      252,\n      254,\n      255,\n      258,\n      263,\n      264,\n      265,\n      266,\n      267,\n      268,\n      270,\n      271,\n      272,\n      273,\n      275,\n      277,\n      278,\n      280,\n      281,\n      282,\n      285,\n      286,\n      288,\n      289,\n      292,\n      293,\n      295,\n      298,\n      299,\n      301,\n      305,\n      306,\n      307,\n      308,\n      309,\n      311,\n      313,\n      314,\n      315,\n      316,\n      317,\n      321,\n      325,\n      328,\n      329,\n      330,\n      338,\n      341,\n      344,\n      345,\n      346,\n      347,\n      349,\n      350,\n      351,\n      354,\n      357,\n      360,\n      361,\n      364,\n      365,\n      367,\n      369,\n      370,\n      372,\n      375,\n      378,\n      379,\n      380,\n      381,\n      383,\n      387,\n      388,\n      390,\n      391,\n      392,\n      393,\n      395,\n      397,\n      398,\n      399\n    ]\n  },\n  "created_date": "2026-10-07",\n  "train_examples": 1663,\n  "test_used": false\n}\n', 'results/diagnostic/qwen_base_bf16_guarded_validation_20261007.json': '{\n  "purpose": "Full validation of beam search with greedy retry on missing EOS; test unused",\n  "test_used": false,\n  "dataset": "google/fleurs km_kh validation",\n  "configuration": {\n    "model": "seanghay/Qwen3-ASR-0.6B-Khmer",\n    "revision": "351f1cc594f084e4899ef6cf7bbb17ec64484080",\n    "adapter": null,\n    "adapter_sha256": null,\n    "manifest_sha256": "ff363da7b8c654243a9808898e878cbdfaeec0f4bd472ab036dc1739bf1d538a",\n    "dtype": "bfloat16",\n    "beams": 3,\n    "batch_size": 4,\n    "force_khmer": true,\n    "max_new_tokens": 512,\n    "source_sha256": "17516f1187a3f4a3cbcbbd7c200042e71da75cdfd62e2c53f371c5b6338d7854",\n    "versions": {\n      "torch": "2.8.0+cu128",\n      "transformers": "4.57.6",\n      "peft": "0.21.0",\n      "qwen-asr": "0.0.6",\n      "datasets": "5.0.1"\n    },\n    "greedy_retry_on_missing_eos": true,\n    "retry_batch_size": 1,\n    "guard_source_sha256": "e305b7169e3ff6d6bdd1aec2d416f15ea01b95fb7db0f92037605a284ef56532",\n    "decoding_source_sha256": "9fcd7af59c52bf5463ecb4d27917ddcdedc38b03a6af8eb6f8ec9fc393bd7ecf"\n  },\n  "hardware": "NVIDIA GeForce RTX 4050 Laptop GPU",\n  "validation_indices": [\n    0,\n    2,\n    3,\n    7,\n    8,\n    9,\n    10,\n    12,\n    14,\n    16,\n    21,\n    22,\n    24,\n    25,\n    26,\n    28,\n    29,\n    30,\n    31,\n    33,\n    35,\n    36,\n    37,\n    38,\n    39,\n    40,\n    42,\n    43,\n    44,\n    46,\n    47,\n    49,\n    50,\n    51,\n    52,\n    53,\n    54,\n    57,\n    58,\n    59,\n    60,\n    62,\n    63,\n    65,\n    68,\n    71,\n    72,\n    73,\n    74,\n    76,\n    77,\n    78,\n    82,\n    83,\n    84,\n    85,\n    86,\n    87,\n    88,\n    90,\n    91,\n    92,\n    94,\n    95,\n    96,\n    97,\n    98,\n    100,\n    102,\n    103,\n    104,\n    105,\n    106,\n    107,\n    108,\n    110,\n    111,\n    112,\n    113,\n    115,\n    116,\n    119,\n    121,\n    125,\n    126,\n    127,\n    128,\n    130,\n    131,\n    134,\n    135,\n    139,\n    140,\n    141,\n    142,\n    145,\n    147,\n    149,\n    150,\n    153,\n    155,\n    156,\n    158,\n    160,\n    161,\n    162,\n    164,\n    165,\n    166,\n    167,\n    169,\n    171,\n    172,\n    173,\n    174,\n    177,\n    178,\n    180,\n    184,\n    190,\n    194,\n    196,\n    197,\n    198\n  ],\n  "examples": [\n    {\n      "validation_index": 0,\n      "reference": "កាលដើមឡើយសម្លៀកបំពាក់គឺធ្ងន់ទទួលឥទ្វិពលដោយវប្បធម៌byzantine នៅភាគខាងកើត។",\n      "hypothesis": "ការដាំឡើងសម្លៀកបំពាក់គឺធ្ងន់ ទទួលឥទ្ធិពលដោយវប្បធម៌ប៉ៃសាន់ចេន នៅភាគខាងកើត",\n      "generated_tokens_including_padding": 194,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 2,\n      "reference": "ភាគខាងជើង ឬតំបន់ sentinel range មានភ្នំខ្ពស់ៗជាងគេនៃទ្វីបអង់តាក់ទិក គឺភ្នំ vinson massif ដែលមានកំពូលកម្ពស់ 4892 ម៉ែត្រ ភ្នំ mount vinson",\n      "hypothesis": "ភាគខាងជើងរាជតំបន់ ស៊ីនីឯណែលរេន មានភ្នំខ្ពស់ៗជាងគេនៃទ្វីបបង្កតែតិច គឺ ភ្នំវេនសុនបានស៊ីស ដែលមានកំពូលកំពស់ បួនពាន់ប្រាំបីរយកៅសិបពីរ ម៉ែត្រ ភ្នំបានវិនសន",\n      "generated_tokens_including_padding": 194,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 3,\n      "reference": "ក្រពើទឹកប្រៃមិនសកម្មក្នុងការរស់នៅក្នុងមហាសមុទ្រនោះទេ ព្រោះជម្រកសំខាន់របស់ពួកវាស្ថិតនៅតាមដងទន្លេភាគខាងជើងនៃទីក្រុង រ៉កខាមថុន",\n      "hypothesis": "ប្រព្រឹត្តទឹកប្រៃ មិនសកម្មនៅកាលរស់នៅក្នុងមហាសមិទ្ធិនោះទេ ព្រោះចំពោះសំខាន់ប៉ុស្តិ៍ពួកវា សុទ្ធតែតាមផ្តុំទន្លេ ភាគខឹងជើង នៃពីក្រុង រ៉ក់ ខាងធន់",\n      "generated_tokens_including_padding": 194,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 7,\n      "reference": "ព្រះពុទ្ធសាសនាទីបេ គឺផ្អែកទៅលើការបង្រៀនរបស់ព្រះពុទ្ធ ប៉ុន្តែវាត្រូវបានពង្រីកដោយមាគ៌ានៃមហាយាននៃសេចក្តីស្រឡាញ់ និងដោយប្រើបច្ចេកទេសជាច្រើនតាមការហាត់យូហ្ការបស់ឥណ្ឌា",\n      "hypothesis": "ព្រះពុទ្ធសាសនាទីបេ គឺផ្អែកទៅលើការបង្រៀនរបស់ព្រះពុទ្ធ ប៉ុន្តែវាត្រូវបានពង្រីកដោយ មាគ៌ានិមហាយាន នៃសេចក្តីត្រឡាញ និងដែលប្រើបច្ចេកទេសជាច្រើនតាមការហាត់យូហ្គារបស់ឥណ្ឌា",\n      "generated_tokens_including_padding": 194,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 8,\n      "reference": "ជាគោលការណ៍ព្រះពុទ្ធសាសនា ទីបេ tibetan គឺមានភាពសាមញ្ញណាស់ វាមានយូហ្កា គុនដាលីនី kundalini yoga ការធ្វើសមាធិ និងមាគ៌ានៃក្តីស្រឡាញ់ទាំងអស់",\n      "hypothesis": "ជាគោលការណ៍ដែលពុទ្ធសាសនាទីបេ គឺមានភាពសាមញ្ញណាស់ វាមានយូហ្គា គុណតាលានី ការធ្វើសមាធិ និងមេកៀននៃក្តីស្រឡាញ់ទាំងអស់",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 9,\n      "reference": "នៅក្នុងប្រព័ន្ធដឹកជញ្ជូនដ៏មមាញឹកនេះ មនុស្សគ្រប់រូបត្រូវបានភ្ជាប់ និងជួយជ្រោមជ្រែងដោយប្រព័ន្ធដឹកជញ្ជូនមួយដែលពឹងផ្អែកលើរថយន្តឯកជន",\n      "hypothesis": "នៅក្នុងប្រព័ន្ធដឹកជញ្ជូនដ៏មមាញឹកនេះ មនុស្សគ្រប់រូបត្រូវបានភ្ជាប់ និងជួយជ្រោមជ្រែងដោយប្រព័ន្ធដឹកជញ្ជូនមួយដែលពឹងផ្អែកលើរថយន្តឯកជន",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 10,\n      "reference": "លទ្ធផលវិភាគអំពីគម្រោងនឹងត្រូវបង្ហោះនៅលើគេហទំព័រសាធារណៈ",\n      "hypothesis": "លទ្ធផលវិភាគអំពីគម្រោងនឹងត្រូវបង្ហោះនៅលើគេហទំព័រសាធារណៈ",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 12,\n      "reference": "ទឹកកំពុងហូរហារលើទំនបទឹកនៅផ្នែកមួយដែលមានប្រវែង 100 ហ្វីត",\n      "hypothesis": "ទឹក កំពុងហូរហៀរលើទំនប់ទឹកនៅភ្នែកមួយដែលមានប្រវែង មួយរយ ហ្វៀត",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 14,\n      "reference": "អ្នកចម្រៀងល្បី ៗ នៅទូទាំងប្រទេសបានបង្ហាញបទចំរៀងផាចាន់ឬចម្រៀងធម្មជាតថា្វយដល់ជើងរបស់ស្ហ្រ៊ី ស្ហ៊ីយ៉ាំ shri shyam",\n      "hypothesis": "នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផាច័ន្ទ ឬចម្រៀងធម្មជាតិថ្វាយដ៏ជើងរបស់ ស៊ីរីក ស៊ីយ៉ាំ Shelley",\n      "generated_tokens_including_padding": 123,\n      "has_eos": true,\n      "primary_has_eos": false,\n      "greedy_fallback": true,\n      "primary_hypothesis": "នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផាច័ន្ទ ឬចម្រៀងធម្មជាតិថ្វាយដល់ជើងរបស់ សិរិស្យាំ នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផាច័ន្ទ ឬចម្រៀងធម្មជាតិថ្វាយដល់ជើងរបស់ សិរិស្យាំ នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផាច័ន្ទ ឬចម្រៀងធម្មជាតិថ្វាយដល់ជើងរបស់ សិរិស្យាំ នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផាច័ន្ទ ឬចម្រៀងធម្មជាតិថ្វាយដល់ជើងរបស់ សិរិស្យាំ នាគចម្រៀងល្បីៗនៅទូទាំងប្រទេស បានបង្ហាញបទចម្រៀងផា�"\n    },\n    {\n      "validation_index": 16,\n      "reference": "អ្នកធម្មជាតិវិទូ និងអ្នកទស្សនវិទូបានផ្តោតលើអត្ថបទបែបបុរាណ និងជាពិសេសលើព្រះគម្ពីរជាភាសាឡាតាំង",\n      "hypothesis": "អ្នកធម្មជាតិវិទូ និងអ្នកទស្សនវិទូបានផ្តោតលើអត្ថបទបែបបុរាណ និងជាពិសេសលើប្រេះគម្ពីរជាពិសេសឡាតាំង",\n      "generated_tokens_including_padding": 512,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 21,\n      "reference": "ប្រសិនបើអ្នកទៅទស្សនាតំបន់អាក់ទិក ឬតំបន់អង់តាក់ទិកក្នុងរដូវរងា អ្នកនឹងជួបប្រទះនូវប៉ូលរាត្រី ដែលមានន័យថា ព្រះអាទិត្យនឹងមិនរះមកត្រង់ចំកណ្តាលមេឃនោះទេ",\n      "hypothesis": "ប្រសិនបើអ្នកទៅទស្សនាតំបន់អាទិត្យ ឬតំបន់អង្គតែអាទិត្យក្នុងរដូវរងារ អ្នកនឹងជួបប្រទះនូវពោធិ៍រាត្រីដែលមានន័យថាព្រះអាទិត្យនឹងមិនរះមកត្រង់ចុងកណ្តាលមេនោះទេ",\n      "generated_tokens_including_padding": 512,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 22,\n      "reference": "នេះបានក្លាយជាទម្លាប់ធម្មតាប៉ុន្តែដែកបានបំផ្លាញដោយវាជ្រេសស៊ីកង់ឈើរបស់រទេះ",\n      "hypothesis": "នេះបានក្លាយជាទម្លាប់ផ្ទាប់ផ្ទាត់ ប៉ុន្តែដៃបានបំផ្លាញដោយវាច្រេសស៊ីកង់ឈើរបស់លទេស",\n      "generated_tokens_including_padding": 512,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 24,\n      "reference": "ខ្ញុំមិនដឹងថាតើអ្នកដឹងវាឬអត់នោះទេ ប៉ុន្តែទំនិញភាគច្រើនដែលមកពីអាមេរិកកណ្តាលបានចូលមកក្នុងប្រទេសនេះដោយគ្មានពន្ធ",\n      "hypothesis": "យើងមិនដឹងថាតើអ្នកដឹងវាឬអត់នោះទេ ប៉ុន្តែទំនិញភាគច្រើនដែលមកពីអាមេរិកកណ្តាលបានចូលមកក្នុងប្រទេសនេះដោយគ្មានពន្ធ",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 25,\n      "reference": "សង្កាត់ភាគច្រើនទទួលបានសេវាពីរថយន្តក្រុងតូចៗរបស់ជប៉ុន ដែលមានផាសុកភាព និងមាំល្អ",\n      "hypothesis": "សង្កាត់ភាគច្រើនទទួលបានសេវាពីរថយន្តក្រុងតូចៗរបស់ជប៉ុន ដែលមានផាសុកភាព និងមាំល្អ",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 26,\n      "reference": "ពួកអារ៉ាប់ក៏បាននាំយកសាសនាអ៊ីស្លាមទៅកាន់ដីគោកដែរ ហើយវាចាប់ផ្ដើមសាយភាយខ្លាំងនៅតំបន់កូម៉ូរូស និងម៉ាយ៉ូតេ",\n      "hypothesis": "ពួកអារ៉ាប់ក៏បាននាំយកសាសនាឥស្លាមទៅកាន់ដីគោកដែរ ហើយវាចាប់ផ្តើមសាយភាយខ្លាំងនៅតំបន់ កុម្មុរស និង ម៉ាយុទេ",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 28,\n      "reference": "គ្រាប់បែក បី គ្រាប់ផ្សេងទៀតបានផ្ទុះឡើងនៅក្បែរអគាររដ្ឋាភិបាលអស់រយៈពេលពីម៉ោងផងដែរ",\n      "hypothesis": "គ្រាប់ប៉ែកបីគ្រាប់ផ្សេងទៀតបានផ្ទុះឡោមនៅខ្មែរអគាររដ្ឋាភិបាលអរិយពេលពីម៉ោងផងដែរ",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 29,\n      "reference": "ក៏ប៉ុន្តែ សញ្ញាភាគច្រើនត្រូវបានបង្ហាញឡើងតែក្នុងភាសាកាតាឡានតែប៉ុណ្ណោះ ពីព្រោះច្បាប់បានកំណត់ថាវាជាភាសាផ្លូវការដំបូងគេ",\n      "hypothesis": "ក៏ប៉ុន្តែសញ្ញាភាគច្រើនត្រូវបានបង្ហាញឡើងតែក្នុងគាសាការតា ឡើងតែប៉ុណ្ណោះ ពីព្រោះច្បាប់បានកំណត់ថា វាជាភាសាផ្លូវការដំបូងគេ",\n      "generated_tokens_including_padding": 143,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 30,\n      "reference": "ការប្រកួតកីឡាធំជាងគេប្រចាំឆ្នាំ ប្រារព្ធធ្វើឡើងនៅខែធ្នូ នៅឯទីលានប៉ូឡូ នៅឡាស ខានីតាស",\n      "hypothesis": "ការប្រកួតកីឡាធំជាងគេប្រចាំឆ្នាំប្រារព្ធធ្វើឡើងនៅខែធ្នូ នៅឯទីលានពោលលោម នៅឡាស់ខណ្ឌីតាស",\n      "generated_tokens_including_padding": 143,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 31,\n      "reference": "ក្រុមអ្នកវិទ្យាសាស្ត្របាននិយាយថារោមរបស់សត្វនេះ គឺមានពណ៌ត្នោតនៅខាងលើ និងមានពណ៌លឿងនៅខាងក្រោម",\n      "hypothesis": "ក្រុមអ្នកវិទ្យាសាស្ត្របាននិយាយថា រោមរបស់សត្វនេះគឺមានពណ៌ត្នោតនៅខាងលើ និងមានពណ៌លឿងនៅខាងក្រោម",\n      "generated_tokens_including_padding": 143,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 33,\n      "reference": "ភាគច្រើននៃជីវិតរបស់គ្រួសារហេព្រើរគឺរស់នៅលើទីវាលបើកចំហរ",\n      "hypothesis": "ភាគច្រើននៃជីវិតរបស់គ្រួសារ ហេប ព្រឹក គឺរស់នៅលើទីវាលបើកចំហ",\n      "generated_tokens_including_padding": 143,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 35,\n      "reference": "បន្ទាប់ពីតក់ស្លុត និងបានឃើញភាពសាហាវឃោរឃៅនៃសង្គ្រាមក្នុងកំឡុងពេលនៃសង្គ្រាមលោកលើកទីមួយ ប្រជាជាតិជាច្រើនមិនចង់ជួបស្ថានភាពបែបនេះទៀតទេនៅពេលអនាគត",\n      "hypothesis": "បន្ទាប់ពីតក់ស្លុត និងបានឃើញភាពសាហាវឃោរឃៅនៃសង្គ្រាមក្នុងកំឡុងពេលដែលសំខាន់ៗប្រជាជាតិជាច្រើនមិនចង់ជួបស្ថានភាពបែបនេះទៀតទេនៅពេលអនាគត",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 36,\n      "reference": "ទន្លេអាម៉ាហ្សូនគឺជាទន្លេដែលធំជាងគេនៅលើភពផែនដីដែលមានទទឹងប្រវែង 6 ម៉ាយល៍",\n      "hypothesis": "ទន្លេអាម៉ាស់ គឺជាទន្លេដ៏អធ្យមជាងគេនៅលើភពផាន្តដីដែលមានទទឹងប្រវែង ដប់មួយ ម៉ែត្រ",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 37,\n      "reference": "ដោយសារតែចម្ងាយឆ្ងាយពីទ្វីបហើយថនិកសត្វមិនអាចធ្វើដំណើរបាន ធ្វើឱ្យសត្វអណ្តើកយក្សក្លាយជាសត្វចិញ្ចឹមដ៏សំខាន់បំផុតនៅកាឡាបាហ្គូស",\n      "hypothesis": "ដោយសារតែចម្ងាយឆ្ងាយពីទ្វារ ហើយថ្លៃចិត្តសត្វម្អិតធ្លាប់បាន ធ្វើឱ្យសត្វម្អិតយក្សក្លាយជាសត្វចិញ្ចឹមដ៏សំខាន់បំផុតនៅកាឡាបារូឌ",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 38,\n      "reference": "សង្កាត់ភាគច្រើនទទួលបានសេវាពីរថយន្តក្រុងតូចៗរបស់ជប៉ុន ដែលមានផាសុកភាព និងមាំល្អ",\n      "hypothesis": "សង្កាត់ភាគច្រើនទទួលបានសេវាពីរថយន្តក្រុងតូចៗរបស់ជប៉ុនដែលមានផាសុកភាព និងមាំល្អ",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 39,\n      "reference": "មិនយូរមិនឆាប់ នៅពេលដែលប្រជាជនថ្មីចាប់ផ្តើមសម្របខ្លួនទៅនឹងបរិស្ថានថ្មីរបស់ខ្លួន ពួកគេចាប់ផ្តើមមើលទៅកាន់តែពុំដូចទៅនឹងប្រជាជនដទៃទៀតនោះឡើយ",\n      "hypothesis": "មិនយូរមិនឆាប់ នៅពេលដែលប្រជាជនថ្មីចាប់ផ្តើមសម្របខ្លួនទៅនឹងបរិស្ថានថ្មីរបស់ខ្លួន ពួកគេចាប់ផ្តើមមើលទៅកាន់តែពុំដូចទៅនឹងប្រជាជនដទៃទៀតនៅឡើយ",\n      "generated_tokens_including_padding": 237,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 40,\n      "reference": "អ៊ីស្រាអែលទាមទារឱ្យមានវត្តមានយោធាជាបន្តនៅជ្រលងភ្នំសម្រាប់រយៈពេលដប់ឆ្នាំ ពេលដែលកិច្ចព្រមព្រៀងត្រូវបានចុះហត្ថលេខា ចំណែកឯប៉ាឡេស្ទីនយល់ព្រមឱ្យមានវត្តមានបែបនេះសម្រាប់រយៈពេលតែប្រាំឆ្នាំ ប៉ុណ្ណោះ",\n      "hypothesis": "ឥស្ត្រអេឡិចត្រូវទាមទារឱ្យមានវត្តមានយោធាជាបន្តនៅជ្រលងភ្នំសម្រាប់រយៈពេល ដប់ ឆ្នាំ ពេលដែលកិច្ចព្រមព្រៀងត្រូវបានចុះអត្តលិកា ចំណែកអបាលីស្ទីនយល់ព្រមឱ្យមានវត្តមានបែបនេះសម្រាប់រយៈពេលតែ ប្រាំ ឆ្នាំប៉ុណ្ណោះ",\n      "generated_tokens_including_padding": 237,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 42,\n      "reference": "យោងតាមសម្តីរបស់ប៉ូលីស អ្នកបើកបររថយន្តដែលបុកអ្នកថតរូបនោះ ប្រហែលជាមិនទទួលរងនូវការចោទប្រកាន់ពីបទឧក្រិដ្ឋនោះទេ",\n      "hypothesis": "យោងតាមសម្តីរបស់ប៉ូលីស អ្នកបើកបរិយានដែលបុកអ្នកថតរូបនោះ ប្រហែលជាមិនទទួលរងនូវការចោទប្រកាន់ពីបទរូបក្រឹត្យនោះទេ",\n      "generated_tokens_including_padding": 237,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 43,\n      "reference": "បុណ្យណូអែល គឺជាថ្ងៃឈប់សម្រាកដ៏សំខាន់មួយនៅក្នុងសាសនាគ្រីស្ទ ហើយត្រូវបានគេអបអរជាថ្ងៃកំណើតរបស់ព្រះគ្រីស្ទ",\n      "hypothesis": "បុណ្យឯក គឺជាថ្ងៃជួបសម្រាកដ៏សំខាន់មួយនៅក្នុងសាសនាគ្រឹស្ត ហើយត្រូវបានគេអបអរជាថ្ងៃកំណើតរបស់ព្រះគ្រឹស្ត",\n      "generated_tokens_including_padding": 237,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 44,\n      "reference": "ភ្លាមៗនោះដែរ មន្រ្តីដែលបំពាក់ដោយសម្លៀកបំពាក់ឧបករណ៍ការពារកុបកម្ម បានចូលទៅកាន់ទីធ្លានោះ ហើយប្រើឧស្ម័នបង្ហូរទឹកភ្នែករុញផ្ទប់អ្នកទោស",\n      "hypothesis": "ភ្លាមៗនោះដែរ មន្ត្រីដែលបំពាក់ដោយសម្លៀកបំពាក់ឧបករណ៍ការពារកុម្ពកម្ម បានចូលទៅកាន់ទីធ្លានោះ ហើយប្រើឧស្ម័នបង្ហូរទឹកភ្នែក រុញផ្ទប់អ្នកទោស",\n      "generated_tokens_including_padding": 236,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 46,\n      "reference": "ស្រាបៀក្នុងស្រុកចម្បង គឺស្រាបៀម៉ាក number one វាមិនមែនជាស្រាបៀស្មុគស្មាញអីនោះទេ ប៉ុន្តែវាជាស្រាបៀបែបរីករាយនិងស្រស់ស្រាយ ស្រាបៀក្នុងស្រុកផ្សេងទៀតត្រូវបានគេហៅថា ម៉ានថា manta",\n      "hypothesis": "សាប៊ីក្នុងសុខចម្បងគឺសាប៊ីម៉ាក់ណឹមវវន់ វាមិនមែនជាសាប៊ីស្មុគស្មាញឯនោះទេ ប៉ុន្តែវាជាសាប៊ីបែបរីករាយ និងសាស្ត្រាយ សាប៊ីក្នុងសុខផ្សេងទៀតត្រូវបានគេហៅថា ម៉ាន់តា",\n      "generated_tokens_including_padding": 236,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 47,\n      "reference": "កីឡាករលេងគ្រីឃីតលោក រ៉ប់ប៊ីន យូថាផា robin uthappa បានវាយគ្រាប់គ្រីឃីតបានពិន្ទុខ្ពស់ជាងគេ គឺបាន 70 ពិន្ទុ រ៉ាន់ ដោយកូនគ្រាប់បាល់ 41គ្រាប់ប៉ុណ្ណោះ ដោយវាយបាន11ដងនៃពិន្ទុបួន ហើយនិង2ដងនៃពិន្ទុប្រាំមួយ",\n      "hypothesis": "កីឡាករលេងក្រីក្រល្លុះ រុបិនយូថផល បិណ្ឌវាយគ្រាប់ក្រីក្របានបន្ទុកខ្ពស់ជាងគេ គឺបានចុះសពន្ធទុកបុរាណ ដោយកូនគ្រាប់បាល់ផ្សែសមួយគ្រាប់ប៉ុណ្ណោះ ដោយវាបានដប់មួយដងនៃពិន្ទុបួន ហើយនិងពីរដងនៃពិន្ទុប្រាំមួយ",\n      "generated_tokens_including_padding": 236,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 49,\n      "reference": "សូមមេត្តាគោរពទីកន្លែងនេះដោយរក្សាសេចក្តីថ្លៃថ្នូរ ភាពស្មឹងស្មាធ និងការគោរពដែលសាកសម កុំលេងសើចអំពីការប្រល័យពូជាសាសន៍ ឬពួកណាស៊ីឲ្យសោះ",\n      "hypothesis": "សូមមេត្តាគោរពទីកន្លែងនេះ ដោយរក្សាសេចក្តីថ្លៃថ្នូរ ភាពស្មឹងស្មាត និងការគោរពដែលស័ក្តិសម កម្លេងសើចអំពីការប្រល័យពូជសាសន៍ ឬពួកណាស៊ីលសោះ",\n      "generated_tokens_including_padding": 236,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 50,\n      "reference": "បណ្តាកោះនៃអាហ្រ្វិកខាងកើតស្ថិតនៅក្នុងមហាសមុទ្រឥណ្ឌានៅតាមឆ្នេរសមុទ្រខាងកើតនៃទ្វីបអាហ្វ្រិក",\n      "hypothesis": "បណ្តាក់សន៍នៃអាហ្វ្រិកខាងកើត ស្ថិតនៅក្នុងមហាសមុទ្រឥណ្ឌានៅតាមឆ្នេរសមុទ្រខាងកើតនៃទ្វីបអាហ្វ្រិក",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 51,\n      "reference": "គាត់បាននិយាយថា គាត់បានផលិតកន្តឹងទ្វារ wifi",\n      "hypothesis": "គាត់បាននិយាយថា គាត់បានផលិតកម្តឹងទ្វារ Wi-Fi",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 52,\n      "reference": "គួបផ្សំនឹងភាពមិនទាក់ទងគ្នា ពាក្យ timbuktu ត្រូវបានគេប្រើជាពាក្យប្រៀបធៀបសម្រាប់ដែនដីនៅឆ្ងាយនិងប្លែកថ្មី",\n      "hypothesis": "គួបសមនឹងភាពមិនទាក់ទងគ្នា ពាក្យ ធីមបុកទូ ត្រូវបានគេប្រើជាពាក្យប្រៀបធៀបសម្រាប់ដែនដីនៅឆ្ងាយ និងប្លែកថ្មី",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 53,\n      "reference": "ជាធម្មតា វគ្គសិក្សានេះ នឹងនិយាយអំពីបញ្ហានានាដែលពិភាក្សានៅក្នុងទីនេះយ៉ាងល្អិតល្អន់ជាងមុន ជាទូទៅមានបញ្ចូលនូវបទពិសោធន៍ជាក់ស្ដែងផងដែរ",\n      "hypothesis": "ជាធម្មតា អារម្មណ៍អាពាហ៍សិក្សានេះនឹងនិយាយអំពីបញ្ហានានា ដែលពិភាក្សានៅក្នុងទីនេះយ៉ាងល្អិតល្អន់ ជាងមុន ជាទូទៅមានបញ្ចូលទម្រង់ពិសោធន៍ជាក់ស្តែងផងដែរ",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 54,\n      "reference": "ចំនួនប្រជាជនដើមមិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "hypothesis": "ជំនាន់ប្រជាជនដ៏មិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "generated_tokens_including_padding": 211,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 57,\n      "reference": "ភាគខាងជើង ឬតំបន់ sentinel range មានភ្នំខ្ពស់ៗជាងគេនៃទ្វីបអង់តាក់ទិក គឺភ្នំ vinson massif ដែលមានកំពូលកម្ពស់ 4892 ម៉ែត្រ ភ្នំ mount vinson",\n      "hypothesis": "ភាគខាងជើង ឬតំបន់សែនឆ្នាំរ៉ែង Range មានភ្នំខ្ពស់ៗជាងគេនៃទ្វីបអង្គតែតិច គឺភ្នំវេនសន ម៉ាស៊ីហ្វ ដែលមិនកម្ពស់កម្ពស់ បួនពាន់ប្រាំបីរយកៅសិបពីរ ម៉ែត្រ ភ្នំម៉ូនវេនសន",\n      "generated_tokens_including_padding": 211,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 58,\n      "reference": "មិនមានព្រៃឈើធំ ៗ នៅក្នុងដែនដីកាណាន canaan ទេ ដូច្នេះ ឈើមានតម្លៃថ្លៃណាស់",\n      "hypothesis": "មិនមានព្រៃឈើធំៗនៅក្នុងដែនដីកាលណាទេ ដូច្នេះឈើមានតម្លៃថ្លៃណាស់",\n      "generated_tokens_including_padding": 211,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 59,\n      "reference": "របាយការណ៍ខ្លះបានបញ្ជាក់ថា ចំនួនអ្នកស្លាប់ជាផ្លូវការមានប្រាំបីនាក់ហើយរបាយការណ៍ជាផ្លូវការបញ្ជាក់ថា មានរហូតដល់ 30 នាក់បានរងរបួស ប៉ុន្តែចំនួនចុងក្រោយមិនទាន់ត្រូវបានគេដឹងនៅឡើយទេ",\n      "hypothesis": "របាយការណ៍ខ្លះបានបញ្ជាក់ថា ចំនួនអ្នកស្លាប់ជាផ្លូវការមានប្រាំបីនាក់ ហើយរបាយការណ៍ជាផ្លូវការបញ្ជាក់ថា មានរហូតដល់សាមសិបនាក់បានរងរបួស ប៉ុន្តែចំនួនជុំក្រោយមិនទាន់ត្រូវបានគេដឹងនៅឡើយទេ",\n      "generated_tokens_including_padding": 211,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 60,\n      "reference": "មានឆ្នេរជាច្រើនកកើតឡើង ដោយសារតែការញែកចេញនៃកំពង់ផែ អក់ឡឹនដ៍ auckland ទាំងពីរ ការពេញនិយមបំផុត គឺស្ថិតនៅក្នុងតំបន់ ឬផ្នែកចំនួនបី",\n      "hypothesis": "មានឆ្នេរជាច្រើនក៏កើតឡើងដោយសារតែការញែកចេញនៃកំពង់ផែអកឡឺលិនទាំងពីរ ការពេញនិយមបំផុតគឺស្ទើរនៅក្នុងដំបង នៅផ្នែកចំនួនបី",\n      "generated_tokens_including_padding": 149,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 62,\n      "reference": "វាត្រូវបានគេគ្រោងនឹងកាត់បន្ថយនៅថ្ងៃអង្គារនេះទេ ប៉ុន្តែវាត្រូវបានរក្សាទុកសម្រេចពេលក្រោយបន្ទាប់ពីសេចក្តីសម្រេចរបស់តុលាការបន្ទាន់",\n      "hypothesis": "វាត្រូវបានគេគ្រោងនិងកាត់បន្ថយនៅថ្ងៃអង្គារនេះទេ ប៉ុន្តែវាត្រូវបានរក្សាទុកសម្រេចពេលក្រោយបន្ទាប់ពីក្តីសម្រេចរបស់តុលាការបន្ទាន់",\n      "generated_tokens_including_padding": 149,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 63,\n      "reference": "ជារឿយៗ ទស្សនៈដែលបានបង្ហាញមានលក្ខណៈមិនលម្អិត ទូទៅ និងសាមញ្ញពេក បើប្រៀបធៀបនឹងព័ត៌មានលម្អិតដែលមាននៅកន្លែងផ្សេងទៀត",\n      "hypothesis": "ជារឿយៗ ទស្សនាដែលបានបង្ហាញមានលក្ខណៈមិនលម្អិតទូទៅ និងសាមញ្ញពេក បើប្រៀបធៀបនឹងព័ត៌មានលម្អិតដែលមាននៅកន្លែងផ្សេងទៀត",\n      "generated_tokens_including_padding": 149,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 65,\n      "reference": "គាត់បាននិយាយថា គាត់បានផលិតកន្តឹងទ្វារ wifi",\n      "hypothesis": "គាត់បាននិយាយថា គាត់បានផលិតកណ្តឹងទ្វារ Wi-Fi",\n      "generated_tokens_including_padding": 149,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 68,\n      "reference": "ការកកិត គឺមានកម្រិតទាបនៅតាមផ្លូវព្រឹលទឹកកក ហើយអ្នកមិនអាចបើកបរបានដូចអ្នកដែលបើកបរនៅលើផ្លូវកៅស៊ូនោះទេ",\n      "hypothesis": "ការកកិត គឺមានកម្រិតទាបនៅតាមផ្លូវព្រឹទ្ធាករ ហើយអ្នកមិនអាចបើកបរបានដូចអ្នកដែលបើកបរនៅលើផ្លូវកៅស៊ូនោះទេ",\n      "generated_tokens_including_padding": 120,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 71,\n      "reference": "សាសនាចម្បងគេនៅក្នុងប្រទេស ម៉ុលដូវ៉ា គឺសាសនាគ្រីស្ទ កាន់វិន័យចាស់",\n      "hypothesis": "សាសនាចម្បងគេនៅក្នុងប្រទេស មូលដូរវ៉ា គឺសាសនាគ្រីស កាន់វិន័យចាស់",\n      "generated_tokens_including_padding": 120,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 72,\n      "reference": "ប្រាក់អាចប្តូរបាននៅធនាគារតែមួយគត់នៅលើកោះនេះ ដែលមានទីតាំងនៅ stanley ទល់មុខហាងលក់ទំនិញ fic west",\n      "hypothesis": "ប្រាក់អាចប្តូរបាននៅធនាគារតែមួយគត់នៅលើកកញ្ញនេះ ដែលមានទីតាំងនៅស្ទីលនេះ ដល់មុខហាងលក់ទំនិញ F I C West",\n      "generated_tokens_including_padding": 120,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 73,\n      "reference": "នៅក្នុងទីធ្លាព្រះវិហារ មានរូបចម្លាក់ជាថ្មម៉ាបដែលគួរឱ្យចាប់អារម្មណ៍នៅលើផ្នូរខ្លះ",\n      "hypothesis": "នៅក្នុងទីធ្លាព្រះវិហារ មានរូបចម្លាក់ជាថ្មមាប ដែលគួរឱ្យចាប់អារម្មណ៍នៅលើម៉ូងខ្លះ",\n      "generated_tokens_including_padding": 120,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 74,\n      "reference": "បុណ្យណូអែល គឺជាថ្ងៃឈប់សម្រាកដ៏សំខាន់មួយនៅក្នុងសាសនាគ្រីស្ទ ហើយត្រូវបានគេអបអរជាថ្ងៃកំណើតរបស់ព្រះគ្រីស្ទ",\n      "hypothesis": "បុណ្យលោអែ គឺជាថ្មឈើសម្រាកដ៏មហន្តរាយមួយនៅក្នុងសាសនាគ្រឹស្ត ហើយត្រូវបានគេអបអរជាថ្មកំណើតរបស់ព្រះគ្រឹស្ត",\n      "generated_tokens_including_padding": 159,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 76,\n      "reference": "ពួកគេកំពុងព្យាយាមធ្វើការកំណត់ថា តើការធ្លាក់បុកនេះមានទំហំធំប៉ុណ្ណា ហើយថាតើផែនដីនឹងរងប៉ះពាល់ដោយសារការធ្លាក់បុកនេះយ៉ាងដូចម្តេចខ្លះ",\n      "hypothesis": "ពួកគេកំពុងព្យាយាមធ្វើការមិត្តថា តើការធ្លាក់មុខនេះមានផលិតផលប៉ុណ្ណា ហើយថតតាមផែនដី និងរងផលប៉ះពាល់ បើសារការធ្លាក់មុខនេះយ៉ាងមិននៅខ្លះ",\n      "generated_tokens_including_padding": 159,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 77,\n      "reference": "អ្នកវិទ្យាសាស្ត្របាននិយាយថា ការផ្ទុះដែលកើតពីការប៉ះទង្គិចគ្នានេះ គឺមានភាពធំសម្បើមណាស់",\n      "hypothesis": "អ្នកវិទ្យាសាស្ត្របាននិយាយថា ការផ្ទុះដែលកើតពីការប៉ះនឹងកិច្ចគ្នានេះ គឺមានភាពធំសម្បើមណាស់",\n      "generated_tokens_including_padding": 159,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 78,\n      "reference": "ទ្រឹស្តីទាំងនេះបង្ហាញថា មនុស្សមានតម្រូវការ និង/ឬ សេចក្តីប្រាថ្នាជាក់លាក់ ដែលត្រូវមាននៅក្នុងខ្លួនរបស់ពួកគេនៅពេលពួកគេធំពេញវ័យ។",\n      "hypothesis": "ទ្រឹស្តីទាំងនេះបង្ហាញថា មនុស្សមានតម្រូវការ និងឬសេចក្តីប្រាថ្នាជាក់លាក់ ដែលត្រូវមាននៅក្នុងខ្លួនរបស់ពួកគេនៅពេលពួកគេធំពេញវ័យ",\n      "generated_tokens_including_padding": 159,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 82,\n      "reference": "ក្រុមអ្នកវិទ្យាសាស្ត្របាននិយាយថារោមរបស់សត្វនេះ គឺមានពណ៌ត្នោតនៅខាងលើ និងមានពណ៌លឿងនៅខាងក្រោម",\n      "hypothesis": "ក្រុមហ៊ុនវិទ្យាសាស្ត្របាននិយាយថា រោមលោកសត្វនេះគឺមានពណ៌ត្នោតនៅខាងលើ និងមានពណ៌លឿងនៅខាងក្រោម",\n      "generated_tokens_including_padding": 109,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 83,\n      "reference": "កម្មវិធីមួយចំនួនមានតំបន់បោះតង់ពិសេសសម្រាប់គ្រួសារដែលមានក្មេងតូចៗ",\n      "hypothesis": "កម្មវិធីមួយចំនួនមានតំបន់បោះតង់ពិសេសសម្រាប់គ្រួសារដែលមានក្មេងតូចៗ",\n      "generated_tokens_including_padding": 109,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 84,\n      "reference": "ចំនួនប្រជាជនដើមមិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "hypothesis": "ចំនួនប្រជាជនដើមមិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "generated_tokens_including_padding": 109,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 85,\n      "reference": "លទ្ធផលវិភាគអំពីគម្រោងនឹងត្រូវបង្ហោះនៅលើគេហទំព័រសាធារណៈ",\n      "hypothesis": "វត្ថុផលវិភាគអំពីគម្លងនឹងត្រូវបង្ខំនៅលើគេហទំព័រសាធារណៈ",\n      "generated_tokens_including_padding": 109,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 86,\n      "reference": "ការបញ្ចេញសូរ គឺមានភាពងាយស្រួលនៅក្នុងភាសាអ៊ីតាលី ដោយសារពាក្យភាគច្រើន ត្រូវបញ្ចេញសូរដូចគ្នាស្ទើរតែទាំងស្រុងទម្រង់សំណេររបស់វា",\n      "hypothesis": "ការបញ្ចេញសូរ គឺមានភាពងាយស្រួលនៅក្នុងភាសាអ៊ីតីលី ដោយសារពាក្យភាគច្រើនត្រូវបញ្ចេញសូរដូចគ្នា ស្ទើរតែទាំងស្រុងទម្រង់សេនារបស់វា",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 87,\n      "reference": "ចំនួនប្រជាជនដើមមិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "hypothesis": "ចំនួនប្រជាជនដើមមិនបានផ្លាស់ប្តូរនោះទេ ពួកគេនៅតែត្រូវការការសម្របខ្លួនដូចតែពីមុនអញ្ចឹង",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 88,\n      "reference": "អ្នកវិទ្យាសាស្ត្រគិតថា ខ្លារខិនដើររកស៊ីចំណី សត្វជាចំណី ដោយការស្រងក្លិន ការហិតក្លិននៅតាមដីដែលវាធ្លាប់នៅ",\n      "hypothesis": "អ្នកវិទ្យាសាស្ត្រគិតថា ខណៈកិនដើររកស៊ីចំណី សត្វជាចំណី ដោយការស្រង់ក្លិន ការហិតក្លិននៅតាមដីដែលវាធ្លាប់នៅ",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 90,\n      "reference": "បុណ្យណូអែល គឺជាថ្ងៃឈប់សម្រាកដ៏សំខាន់មួយនៅក្នុងសាសនាគ្រីស្ទ ហើយត្រូវបានគេអបអរជាថ្ងៃកំណើតរបស់ព្រះគ្រីស្ទ",\n      "hypothesis": "ពូនឯកគឺជាថ្ងៃចុះសម្រាកដ៏សំខាន់មួយនៅក្នុងសាសនាគ្រិស្ត ហើយត្រូវបានគេអបអរជាថ្ងៃកំណត់របស់ព្រះគ្រឹះ",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 91,\n      "reference": "ផ្កាយរណបនៅក្នុងលំហអាកាស ទទួលបានសញ្ញា ហើយបន្ទាប់មកបញ្ជូនសញ្ញានោះត្រឡប់មកផែនដីវិញ ស្ទើរតែឥតដាច់",\n      "hypothesis": "កាយរណប នៅក្នុងលំហអាកាសទទួលបានសញ្ញា ហើយបន្ទាប់មកបញ្ជូនសញ្ញានោះត្រឡប់មកផែនដីវិញ ស្ទើរតែឥតដាច់",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 92,\n      "reference": "នៅពេលយប់ងងឹត ពួកគេអាចមើលឃើញច្បាស់ ហើយក៏ធ្វើដំណើរបានដោយរលូនផងដែរ សត្វខ្លារខិនប្រមាញ់រកចំណីដោយលាក់ខ្លួនពួនចាំចាប់ចំណី",\n      "hypothesis": "នាពេលយប់ងងឹត ពួកគេអាចមើលឃើញច្បាស់ ហើយក៏ធ្វើដំណើរបានដោយរលូនផងដែរ សត្វខ្លារខិនប្រមាញ់រកចំណីដោយលាក់ខ្លួនពួនចាំចាប់ចំណី",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 94,\n      "reference": "មានត្រឹមតែមេដាយចំនួន18ប៉ុណ្ណោះក្នុងមួយថ្ងៃ ប្រទេសជាច្រើនបរាជ័យក្នុងការដណ្តើមយកមេដាយទាំងនោះ",\n      "hypothesis": "មានត្រឹមតែមេដាយចំនួន ដប់ប្រាំបី ប៉ុណ្ណោះក្នុងមួយថ្ងៃ ប្រទេសជាច្រើនបរាជ័យក្នុងការនាំយកមេដាយទាំងនោះ",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 95,\n      "reference": "សាហ្វារីស safaris ប្រហែលជាចំណុចទាក់ទាញភ្ញៀវទេសចរណ៍ដ៏អស្ចារ្យបំផុតនៅអាហ្វ្រិក និងជាការបញ្ជាក់ប្រាប់ដល់ភ្ញៀវទេសចរជាច្រើនទៀត",\n      "hypothesis": "សាស្វារីស ប្រហារជាចំណុចទាក់ទាញព្យាបាលទេសចរណ៍ដ៏អស្ចារ្យបំផុតនៅអាហ្វ្រិក នៅជាការបញ្ជាក់តាប់ដល់ភ្ញៀវទេសចរជាច្រើនទៀត",\n      "generated_tokens_including_padding": 145,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 96,\n      "reference": "ជាគោលការណ៍ព្រះពុទ្ធសាសនា ទីបេ tibetan គឺមានភាពសាមញ្ញណាស់ វាមានយូហ្កា គុនដាលីនី kundalini yoga ការធ្វើសមាធិ និងមាគ៌ានៃក្តីស្រឡាញ់ទាំងអស់",\n      "hypothesis": "ជាគោលការណ៍ ព្រះពុទ្ធសាសនាទីបេ គឺមានភាពសាមញ្ញណាស់ វាមានយូហ្គាគុណដាលីនី ការធ្វើសមាធិ និងមាគ៌ានិក្តីស្រឡាញ់ទាំងអស់",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 97,\n      "reference": "សត្វខ្លះដូចជា ដំរី និង សត្វកវែង ព្យាយាមចូលមកជិតឡាន ដូច្នេះសម្ភារៈតាមបទដ្ឋាន គឺអំណោយផលក្នុងការមើលទេសភាពបានល្អ",\n      "hypothesis": "សត្វខ្លះដូចជាដំរី និងសត្វកវែងព្យាយាមចូលមកជិតឡាន ដូច្នេះសម្ភារតាមបទដ្ឋានគឺអំណោយផលក្នុងការមើលទេសភាពបានល្អ",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 98,\n      "reference": "អំណាចលើគ្រប់វិស័យរបស់ខ្លួន បានជះឥទ្ធិពលដល់មនុស្សគ្រប់គ្នាចាប់ពីស្តេច រហូតដល់មនុស្សសាមញ្ញ",\n      "hypothesis": "អំណាចលើគ្រប់វិស័យរបស់ខ្លួនបានជះឥទ្ធិពលដល់មនុស្សគ្រប់គ្នា ចាប់ពីស្តេច រហូតដល់មនុស្សសម្តែង",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 100,\n      "reference": "បន្ទាប់មក ឡាក់ខាស៊ីងបាននាំមុខក្នុងការច្រៀងចម្រៀងផាចានស៍",\n      "hypothesis": "បន្ទាប់មក ឡាក់ ខាស៊ីង បាននាំមកក្នុងការច្រៀងចម្រៀងផាចាន",\n      "generated_tokens_including_padding": 132,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 102,\n      "reference": "នៅក្នុងប្រព័ន្ធដឹកជញ្ជូនដ៏មមាញឹកនេះ មនុស្សគ្រប់រូបត្រូវបានភ្ជាប់ និងជួយជ្រោមជ្រែងដោយប្រព័ន្ធដឹកជញ្ជូនមួយដែលពឹងផ្អែកលើរថយន្តឯកជន",\n      "hypothesis": "នៅក្នុងប្រព័ន្ធដឹកជញ្ជូនដ៏មមាញឹកនេះ មនុស្សគ្រប់រូបត្រូវបានភ្ជាប់ និងជួយជ្រោមជ្រែងដោយប្រព័ន្ធដឹកជញ្ជូនមួយដែលពឹងផ្អែកលើរថយន្តឯកជន",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 103,\n      "reference": "បើទោះបីជាមានមនុស្សបីនាក់ស្ថិតនៅក្នុងផ្ទះនៅពេលរថយន្តបើកបុកក្ដី ក៏ប៉ុន្តែពុំមាននរណាម្នាក់រងរបួសនោះទេ",\n      "hypothesis": "បើទោះបីជាមានមនុស្សបីនាក់ស្ថិតនៅក្នុងផ្ទះនៅពេលរថយន្តបើកបុកក្តី ក៏ប៉ុន្តែពុំមាននរណាម្នាក់រងរបួសនោះទេ",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 104,\n      "reference": "ផ្លូវទឹកនៅក្នុងប្រទេសអាចជាប្រធានបទដ៏ល្អមួយសម្រាប់ដំណើរកម្សាន្តថ្ងៃឈប់សម្រាក",\n      "hypothesis": "ផ្លូវទឹកនៅក្នុងប្រទេសអាចជាប្រធានបទដ៏ល្អមួយសម្រាប់ដំណើរកម្សាន្តថ្ងៃឈប់សម្រាក",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 105,\n      "reference": "ក៏ប៉ុន្តែ សញ្ញាភាគច្រើនត្រូវបានបង្ហាញឡើងតែក្នុងភាសាកាតាឡានតែប៉ុណ្ណោះ ពីព្រោះច្បាប់បានកំណត់ថាវាជាភាសាផ្លូវការដំបូងគេ",\n      "hypothesis": "ក៏មិនតែសញ្ញាភាគច្រើនទៅបានបង្ហាញឡើងតែក្នុងភាសាកាត់តាឡានតែប៉ុណ្ណោះពីព្រោះច្បាប់បានកំណត់ថាវិជ្ជាភាសាផ្លូវការដំបូងគេ",\n      "generated_tokens_including_padding": 156,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 106,\n      "reference": "ក៏ប៉ុន្តែ សញ្ញាភាគច្រើនត្រូវបានបង្ហាញឡើងតែក្នុងភាសាកាតាឡានតែប៉ុណ្ណោះ ពីព្រោះច្បាប់បានកំណត់ថាវាជាភាសាផ្លូវការដំបូងគេ",\n      "hypothesis": "ក៏ប៉ុន្តែសញ្ញាភាគច្រើនទៅបានបង្ហាញឡើងតែក្នុងភាសាកាត់តាឡានតែប៉ុណ្ណោះ ពីព្រោះច្បាប់បានកំណត់ថាវាជាភាសាហ្វិកការណ៍ដំបូងគេ",\n      "generated_tokens_including_padding": 142,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 107,\n      "reference": "ផ្លូវទឹកនៅក្នុងប្រទេសអាចជាប្រធានបទដ៏ល្អមួយសម្រាប់ដំណើរកម្សាន្តថ្ងៃឈប់សម្រាក",\n      "hypothesis": "ផ្លូវទឹកនៅក្នុងប្រទេសអាចជាប្រធានបទដ៏ល្អមួយសម្រាប់ដំណើរកម្សាន្តថ្ងៃឈប់សម្រាក",\n      "generated_tokens_including_padding": 142,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 108,\n      "reference": "អ្នកវិទ្យាសាស្ត្របាននិយាយថា ការផ្ទុះដែលកើតពីការប៉ះទង្គិចគ្នានេះ គឺមានភាពធំសម្បើមណាស់",\n      "hypothesis": "អ្នកវិទ្យាសាស្ត្របាននិយាយថា ការផ្ទុះដែលកាលពីកាលប៉ះទង្គិចគ្នានេះ គឺមានភាពធុងសម្បើមណាស់",\n      "generated_tokens_including_padding": 142,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 110,\n      "reference": "ផ្កាយរណបនៅក្នុងលំហអាកាស ទទួលបានសញ្ញា ហើយបន្ទាប់មកបញ្ជូនសញ្ញានោះត្រឡប់មកផែនដីវិញ ស្ទើរតែឥតដាច់",\n      "hypothesis": "កែលម្អនៅក្នុងលំហ ឱ្យការទទួលបានសញ្ញា ហើយបន្ទាប់មកបញ្ជូនសញ្ញានោះត្រឡប់មកផែនដីវិញស្ទើរតែឥតដាច់",\n      "generated_tokens_including_padding": 142,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 111,\n      "reference": "ទូរសព្ទផ្កាយរណប ជាទូទៅមិនមែនជាការជំនួសទូរសព្ទចល័តទេព្រោះអ្នកត្រូវតែនៅខាងក្រៅអគារ ដោយអាចឲ្យផ្កាយរណបមើលបានច្បាស់ ដើម្បីហៅទូរសព្ទចេញបាន",\n      "hypothesis": "ទូរស័ព្ទកែរណ្តៅ ជាទូទៅមិនមែនជាការជំនួសទូរស័ព្ទចល័តទេ ព្រោះអ្នកត្រូវតែនៅខាងក្រៅអគារ ដោយហាត់ឱ្យកែរណ្តៅមើលបានច្បាស់ ដើម្បីហៅទូរស័ព្ទចេញបាន",\n      "generated_tokens_including_padding": 209,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 112,\n      "reference": "ក្រុមហ៊ុននាំសំបុត្រទទួលបានប្រាក់ខែខ្ពស់សម្រាប់ការដឹកជញ្ជូនទំនិញយ៉ាងឆាប់រហ័ស ជារឿយៗ ពេលវេលាពិតជាសំខាន់ណាស់ជាមួយឯកសារអាជីវកម្មផ្សេងៗ ទំនិញ ឬ គ្រឿងបន្លាស់សម្រាប់ការជួសជុលបន្ទាន់",\n      "hypothesis": "ក្រុមឧណ្ហៗសំបុត្រទទួលបានប្រាក់ខែខ្ពស់សម្រាប់ការដឹកជញ្ជូនទំនិញយ៉ាងឆាប់រហ័ស ជារឿយៗពេលវេលាបានជាសំខាន់ណាស់ ជាមួយឯកសារអាជីវកម្មផ្សេងៗ ទំនិញ ឬគ្រឿងបន្លាស់សម្រាប់ការជួសជុលបន្ទាន់",\n      "generated_tokens_including_padding": 209,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 113,\n      "reference": "ផលិតផលសិប្បកម្មអាចត្រូវបានចាត់ទុកថាជាវត្ថុបុរាណ ទោះបីជាវាមានអាយុតិចកាលជាងទំនិញស្រដៀងគ្នាដែល ផលិតបានច្រើនក៏ដោយ",\n      "hypothesis": "ផលិតផលសិប្បកម្មអាចត្រូវបានចាត់ទុកថាជាវត្ថុបុរាណ ទោះបីជាវាមានអាយុតិចកាលចៀនទំនិញសំដៅគ្នាដែលផលិតបានច្រើនក៏ដោយ",\n      "generated_tokens_including_padding": 209,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 115,\n      "reference": "លោកវេជ្ជបណ្ឌិត moll គិតថា អ្នកជំងឺមួយចំនួនប្រហែលបានឆ្លងពួកសត្វល្អិតនៅក្នុងមន្ទីរពេទ្យ ហើយយ៉ាងហោចក្នុងនោះ ក៏មានបុគ្គលិកសុខាភិបាលពីរនាក់នៅក្នុងមន្ទីរពេទ្យដែរដែលបានឆ្លងជំងឺ",\n      "hypothesis": "លំហិចម្តិចមូលគណៈ អ្នកជំងឺមួយចំនួនប្រហែលបានឆ្លងពួកសាលានៅក្នុងមន្ទីរពេទ្យ ហើយយ៉ាងហោចក្នុងនោះក៏មានបុគ្គលិកសក្ការបាល់ពីរនាក់នៅក្នុងមន្ទីរពេទ្យដែរដែលបានឆ្លងជំងឺ",\n      "generated_tokens_including_padding": 209,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 116,\n      "reference": "កាលពីរាប់ពាន់ឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា aristarchus បានពោលថា ប្រព័ន្ធសូឡាគឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "hypothesis": "កាលពីរប៉ុន្មានឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា អារ៉េស្តថឺស បានពោលថា ប្រព័ន្ធសូឡាគឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "generated_tokens_including_padding": 166,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 119,\n      "reference": "ជម្រើសដែលកាន់តែពេញនិយមសម្រាប់អ្នក ដែលមានផែនការទុកមួយឆ្នាំចោល រៀនចប់ទុកមួយឆ្នាំទំនេរ គឺការធ្វើដំណើរនិងរៀនអ្វីប្លែកថ្មី",\n      "hypothesis": "ជម្រើសដែលកាន់តែពេញនិយមសម្រាប់អ្នកដែលមានផែនការទុកមួយឆ្នាំចោល ឬចាប់ទុកមួយឆ្នាំទំនេរ គឺ ការធ្វើដំណើរ និងរៀនអ្វីប្លែកថ្មី",\n      "generated_tokens_including_padding": 166,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 121,\n      "reference": "ការលាតសន្ធឹងរវាងទំនប់ ភ័ញមែរៀន point marion និង ហ្វ៊ែរម៉ន់ថ៍ fairmont ដែលបង្ហាញពីស្ថានភាពបើកបរដ៏លំបាកបំផុតនៅលើមហាវិថី បាហ្វហ្វាឡូ ភីតស៍ប៊ើហ្វ buffalo-pittsburgh ដែលត្រូវបានឆ្លងកាត់ជាញឹកញាប់តាមរយៈដីរាបស្មើ",\n      "hypothesis": "ការលេចសន្ធឹងរវាងទំនប់ ផុយមេរៀន និងហ្វេមាន់ ដែលបង្ហាញពីសន្តិភាពបើកបរដ៏លំបាកបំផុតនៅលើមហាវិថី Buffalo Pizza ដែលត្រូវបានឆ្លងកាត់ជាញឹកញាប់តាមរយៈដីរាបស្មើ",\n      "generated_tokens_including_padding": 166,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 125,\n      "reference": "កម្មវិធីមួយចំនួនមានតំបន់បោះតង់ពិសេសសម្រាប់គ្រួសារដែលមានក្មេងតូចៗ",\n      "hypothesis": "ការមានវិធីមួយចំនួនមានតំបន់បោះតង់ពិសាសម្រាប់គ្រួសារដែលមានក្មេងតូចៗ",\n      "generated_tokens_including_padding": 166,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 126,\n      "reference": "លទ្ធផលវិភាគអំពីគម្រោងនឹងត្រូវបង្ហោះនៅលើគេហទំព័រសាធារណៈ",\n      "hypothesis": "លទ្ធផលវិភាគអំពីគម្រោងនឹងត្រូវបង្ហោះនៅលើគេហទំព័រសាធារណៈ",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 127,\n      "reference": "ជម្រើសដែលកាន់តែពេញនិយមសម្រាប់អ្នក ដែលមានផែនការទុកមួយឆ្នាំចោល រៀនចប់ទុកមួយឆ្នាំទំនេរ គឺការធ្វើដំណើរនិងរៀនអ្វីប្លែកថ្មី",\n      "hypothesis": "ជម្រើសដែលកាន់តែពេញនិយមសម្រាប់អ្នកដែលមានផាសុកការទុកមួយឆ្នាំចោល ឬចប់ទុកមួយឆ្នាំទំនេរ គឺការធ្វើដំណើរ និងរៀនអ្វីប៉ុណ្ណោះប្លែកថ្មី",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 128,\n      "reference": "អ្នកជំងឺធ្លាប់បានទៅប្រទេសនីហ្សេរីយ៉ា ដែលនៅប្រទេសនេះបានកើតមានករណីវីរុស អេបូឡាមួយចំនួន",\n      "hypothesis": "អ្នកជំងឺធ្លាប់បានទៅប្រទេស និស្សិរិយា ដែលនៅប្រទេសនេះបានកើតមានកាតិនីរូប អេបូឡា មួយចំនួន",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 130,\n      "reference": "អាជ្ញាធរមូលដ្ឋាន កំពុងផ្ដល់ការជូនដំណឹងដល់ពលរដ្ឋនៅក្បែររោងចក្រនោះ ឲ្យស្ថិតនៅក្នុងផ្ទះ បិទម៉ាស៊ីនត្រជាក់ មិនឲ្យទទួលទានទឹករ៉ូប៊ីណេឡើយ",\n      "hypothesis": "អាជ្ញាធរមូលដ្ឋានកំពុងតភក្តិជូនអំណឹងដល់ពលរដ្ឋនៅក្បែររោងចក្រនោះ ឱ្យស្ថិតនៅក្នុងផ្ទះបិទម៉ាស៊ីនត្រជាក់ មិនឱ្យទទួលទានទឹករ៉ូប៊ីនេឡើយ",\n      "generated_tokens_including_padding": 160,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 131,\n      "reference": "ការជិះស្គីគឺជាសកម្មភាពធ្វើដំណើរដ៏ចម្បងមួយដែលពេញនិយមសម្រាប់មនុស្សជាច្រើនដែលជួនកាលគេហៅថា ស្គីបាំ គម្រោងវិស្សមកាលទាំងមូលជុំវិញការជិះស្គីនៅទីតាំងដ៏ពិសេសមួយ",\n      "hypothesis": "ការជិះស្គី គឺជាសកម្មភាពធ្វើដំណើរដ៏ចម្បងមួយដែលពេញនិយម សម្រាប់មនុស្សជាច្រើនដែលជួនកាលគេហៅថា ស្គីបាប់គម្រោង វិស្វកាលទាំងមូលជុំវិញការជិះស្គីនៅទីតាំងដ៏ពិសេសមួយ",\n      "generated_tokens_including_padding": 192,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 134,\n      "reference": "មានត្រឹមតែមេដាយចំនួន18ប៉ុណ្ណោះក្នុងមួយថ្ងៃ ប្រទេសជាច្រើនបរាជ័យក្នុងការដណ្តើមយកមេដាយទាំងនោះ",\n      "hypothesis": "មានត្រឹមតែមេដាយចំនួន ដប់ប្រាំបី ប៉ុណ្ណោះក្នុងមួយថ្ងៃ ប្រទេសជាច្រើនបរាជ័យក្នុងការដណ្តើមយកមេដាយទាំងនោះ",\n      "generated_tokens_including_padding": 192,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 135,\n      "reference": "ភាគច្រើននៃជីវិតរបស់គ្រួសារហេព្រើរគឺរស់នៅលើទីវាលបើកចំហរ",\n      "hypothesis": "ភាគឆ្លើយនៃជីវិតរបស់គ្រួសារ ហេតុប្រើ គឺរស់នៅលើទីវាលបើកចំហ",\n      "generated_tokens_including_padding": 192,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 139,\n      "reference": "រឿងបែបនេះបានក្លាយជាវិន័យដាច់ដោយឡែក ដែលផ្តោតលើការផ្តល់ដំណោះស្រាយចំពោះបញ្ហាជីវិតពិត",\n      "hypothesis": "រូងបែបនេះបានក្លាយជាវិន័យដាច់ដោយឡែក ដែលផ្តោតលើការផ្តល់ដំណោះស្រាយចំពោះបញ្ហាជីវិតពិត",\n      "generated_tokens_including_padding": 192,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 140,\n      "reference": "ទឹកកំពុងហូរហារលើទំនបទឹកនៅផ្នែកមួយដែលមានប្រវែង 100 ហ្វីត",\n      "hypothesis": "ទឹកកំពង់ហូរហាលលើទំនប់ទឹកនៅផ្នែកមួយដែលមានប្រវែង មួយរយ វីដេអូ",\n      "generated_tokens_including_padding": 226,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 141,\n      "reference": "បើទោះបីជាមានការប្រើប្រាស់យ៉ាងទូលំទូលាយក្ដី ជាពិសេសក្នុងចំណោមពលរដ្ឋផ្សេងពីរូម៉ានី ពាក្យថា gypsy ជារឿយត្រូវគិតថាមានន័យមិនល្អ ដោយសារវាពាក់ព័ន្ធផ្នត់គំនិតអវិជ្ជមាន និងទស្សនៈមិនត្រឹមត្រូវរបស់ពលរដ្ឋរូម៉ានី",\n      "hypothesis": "បើទោះបីជាមានការប្រើប្រាស់យ៉ាងទូលំទូលាយក្តី ជាពិសេសក្នុងចំណោមពលរដ្ឋផ្សេងពីរមណី ពាក្យ Cambodia ជារោយត្រូវគិតថា មានន័យមិនល្អ ដោយសារវាពាក់ព័ន្ធផ្នត់គំនិតអវិជ្ជមាន និងទស្សនៈមិនត្រឹមត្រូវរបស់ពលរដ្ឋរមណី",\n      "generated_tokens_including_padding": 226,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 142,\n      "reference": "ទោះជាយ៉ាងណា រាល់ការផ្លាស់ប្តូរឬការស្នើសុំគប្បីឆ្លងកាត់ភ្នាក់ងារធ្វើដំណើរជាមុន ហើយមិនត្រូវធ្វើឡើងដោយផ្ទាល់តាមរយៈសណ្ឋាគារនោះទេ",\n      "hypothesis": "ទោះជាយ៉ាងណា រាល់ការផ្លាស់ប្តូរ ឬការស្នើសុំគុម្ពបីឆ្លងកាត់ភ្នាក់ងារធ្វើដំណើរជាមុន ហើយមិនត្រូវធ្វើឡើងដោយផ្ទាល់តាមរយៈសណ្ឋាគារនោះទេ",\n      "generated_tokens_including_padding": 226,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 145,\n      "reference": "រឿងបែបនេះបានក្លាយជាវិន័យដាច់ដោយឡែក ដែលផ្តោតលើការផ្តល់ដំណោះស្រាយចំពោះបញ្ហាជីវិតពិត",\n      "hypothesis": "រឿងបែបនេះបានក្លាយជាវិន័យដៃដ៏លៃដែលផ្តោតលើការផ្តល់ដំណោះស្រាយចំពោះបញ្ហាជីវិតពិត",\n      "generated_tokens_including_padding": 226,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 147,\n      "reference": "កាលពីរាប់ពាន់ឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា aristarchus បានពោលថា ប្រព័ន្ធសូឡាគឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "hypothesis": "កាលពីរាប់ពាន់ឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា អារ៉េស្តាឈូក បានពោលថា ប្រព័ន្ធសូឡាគឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 149,\n      "reference": "អាជ្ញាធរមូលដ្ឋាន កំពុងផ្ដល់ការជូនដំណឹងដល់ពលរដ្ឋនៅក្បែររោងចក្រនោះ ឲ្យស្ថិតនៅក្នុងផ្ទះ បិទម៉ាស៊ីនត្រជាក់ មិនឲ្យទទួលទានទឹករ៉ូប៊ីណេឡើយ",\n      "hypothesis": "អាជ្ញាធរមូលដ្ឋានកំពុងផ្តល់ការជូនដំណឹងដល់ពលរដ្ឋដើម្បីរ៉ុញចាក់នោះឱ្យស្ថិតក្នុងផ្ទះ ទឹកដែលសិនត្រជាក់ មិនឱ្យទទួលទានទឹករកពិន័យឡើយ",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 150,\n      "reference": "អ្នកវិទ្យាសាស្ត្រគិតថា ខ្លារខិនដើររកស៊ីចំណី សត្វជាចំណី ដោយការស្រងក្លិន ការហិតក្លិននៅតាមដីដែលវាធ្លាប់នៅ",\n      "hypothesis": "អ្នកវិទ្យាសាស្ត្រគិតថា ខ្លារខិនដើររកស៊ីចំណី ដោយការស្រង់ក្លិន ការហិតក្លិននៅតាមដីដែលវាធ្លាប់នៅ",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 153,\n      "reference": "នៅទីបំផុត កង់ឈើត្រូវបានជំនួសដោយកង់ដែក នៅឆ្នាំ 1767 ផ្លូវដែកពេញលេញដំបូងគេ ត្រូវបានដាក់ឲ្យដំណើរការ",\n      "hypothesis": "នៅទីបំផុត កង់ឈើត្រូវបានចំណូលដោយកង់ដែក នៅឆ្នាំមួយពាន់ប្រាំពីររយហុកសិបប្រាំពីរ ផ្លូវដែកពេញលេញដំបូងគេទៅបានដាក់ឱ្យដំណើរការ",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 155,\n      "reference": "ជារឿយៗ ទស្សនៈដែលបានបង្ហាញមានលក្ខណៈមិនលម្អិត ទូទៅ និងសាមញ្ញពេក បើប្រៀបធៀបនឹងព័ត៌មានលម្អិតដែលមាននៅកន្លែងផ្សេងទៀត",\n      "hypothesis": "ជារឿយៗ ទស្សនៈដែលបានបង្ហាញមានលក្ខណៈមិនលម្អិតទូទៅ និងសាមញ្ញពេក បើពិធី និងព័ត៌មានលម្អិតដែលមាននៅកន្លែងផ្សេងទៀត",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 156,\n      "reference": "ខ្ញុំមិនដឹងថាតើអ្នកដឹងវាឬអត់នោះទេ ប៉ុន្តែទំនិញភាគច្រើនដែលមកពីអាមេរិកកណ្តាលបានចូលមកក្នុងប្រទេសនេះដោយគ្មានពន្ធ",\n      "hypothesis": "ខ្ញុំមិនដឹងថាតើអ្នកដឹងវាឬអត់នោះទេ ប៉ុន្តែទំនិញភាគច្រើនដែលមកពីអាមេរិកកណ្តាលបានចូលមកក្នុងប្រទេសនេះដោយគ្មានប៉ុន",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 158,\n      "reference": "ក្រុមអ្នកស្រាវជ្រាវបានលើកឡើងថា ទោះបីនេះជាកន្ទុយរបស់ដាយណូស័រវ័យក្មេងក៏ដោយតែគំរូបង្ហាញពីរោមដាយណូស័រពេញវ័យហើយក៏មិនមែនជារោមមាន់ដែរ",\n      "hypothesis": "ក្រុមអ្នកស្រាវជ្រាវបានដឹកឡើងថា ទោះបីនេះជាកន្ទុយរបស់អ្នកនៅសើរវ័យក្មេងក៏ដោយ តែគំរូបង្ហាញពីរោមដែលដុះសើរវិញវ័យ ហើយក៏មិនមានជារោមមុនដែរ",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 160,\n      "reference": "បន្ទាប់ពីតក់ស្លុត និងបានឃើញភាពសាហាវឃោរឃៅនៃសង្គ្រាមក្នុងកំឡុងពេលនៃសង្គ្រាមលោកលើកទីមួយ ប្រជាជាតិជាច្រើនមិនចង់ជួបស្ថានភាពបែបនេះទៀតទេនៅពេលអនាគត",\n      "hypothesis": "បន្ទាប់ពីតក់ស្លុត និងបានឃើញភាពសាហាវឃោរឃៅនៃសង្គ្រាមក្នុងកំឡុងពេលនៃសង្គ្រាមលោកលើកទីមួយ ប្រជាជាតិជាច្រើនមិនចង់ជួបស្ថានភាពបែបនេះទៀតទេនៅពេលអនាគត",\n      "generated_tokens_including_padding": 173,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 161,\n      "reference": "ប៉ូលីសបាននិយាយថា សាកសពដែលបានរកឃើញបាននៅទីនោះ រយៈពេលមួយថ្ងៃហើយ",\n      "hypothesis": "ប៉ូលិសបាននិយាយថា សាកសពដែលបានរកឃើញបាននៅទីនោះរយៈពេលមួយថ្ងៃហើយ",\n      "generated_tokens_including_padding": 163,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 162,\n      "reference": "កាលពីរាប់ពាន់ឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា aristarchus បានពោលថា ប្រព័ន្ធសូឡាគឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "hypothesis": "កាលពីរប៉ុន្មានឆ្នាំមុន បុរសម្នាក់ដែលមានឈ្មោះថា អារី សីតឺសិស្ស ប័ណ្ណពរថាប្រព័ន្ធសូឡា គឺធ្វើដំណើរជុំវិញព្រះអាទិត្យ",\n      "generated_tokens_including_padding": 163,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 164,\n      "reference": "នៅប្រទេសអ៊ីរ៉ង់ គ្រោះថ្នាក់ជើងអាកាសគឺជារឿងធម្មតាព្រោះយន្តហោះចាស់ៗ ហើយការថែទាំមិនបានល្អសម្រាប់ទាំងប្រតិបត្តិការស៊ីវិល និងប្រតិបត្តិការយោធា",\n      "hypothesis": "នៅប្រទេសអ៊ីរ៉ុង គោលដៅជាងឱកាសគឺជារឿងធម្មតា ព្រោះយន្តហោះចាស់ៗ ហើយការថែទាំមិនអានល្អ ព្រមទាំងប្រតិបត្តិការស៊ីវិល និងប្រតិបត្តិការយោធា",\n      "generated_tokens_including_padding": 163,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 165,\n      "reference": "រឿងនោះហាក់ដូចជាធ្វើឲ្យខ្ញុំមិនយល់សោះ វាពិតជាមិនយុត្តិធម៌ទេ",\n      "hypothesis": "រឿងនោះហាក់ដូចជាធ្វើឱ្យខ្ញុំមិនយល់សោះ វាពិតជាមិនយុត្តិធម៌ទេ",\n      "generated_tokens_including_padding": 163,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 166,\n      "reference": "គ្រាប់បែក បី គ្រាប់ផ្សេងទៀតបានផ្ទុះឡើងនៅក្បែរអគាររដ្ឋាភិបាលអស់រយៈពេលពីម៉ោងផងដែរ",\n      "hypothesis": "គ្រាប់បែកបីគ្រាប់ផ្សេងទៀតបានផ្ទុះឡើងនៅក្បែរអគាររដ្ឋាភិបាលអស់រយៈពេលពីរម៉ោងផងដែរ",\n      "generated_tokens_including_padding": 150,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 167,\n      "reference": "វាក៏ពុំមានសិទ្ធិអំណាច ដើម្បីបដិសេធច្បាប់ពន្ធ និងពន្ធគយរវាងរដ្ឋនីមួយៗនោះដែរ",\n      "hypothesis": "វាក៏ពុំមានសិទ្ធិអំណាចដើម្បីបរិស័ទច្បាប់ពន្ធ និងពន្ធគយរវាងរដ្ឋនីមួយៗនោះដែរ",\n      "generated_tokens_including_padding": 150,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 169,\n      "reference": "ពិន្ទុចុងក្រោយគឺជាពិន្ទុឈ្នះមួយពិន្ទុ គឺ២១ទល់និង២០ នៅចុងបញ្ចប់នៃការប្រកួតជ័យលាភីហ្គេមall blacksទាំង១៥។",\n      "hypothesis": "ពិន្ទុចុងក្រោយ គឺជាពិន្ទុឆ្នាំ មួយពាន់បួនរយ គឺ ម្ភៃមួយ ទៅ ម្ភៃ នៃចុងបញ្ចប់នៃការប្រកួតជ័យលាភីហ្គេម ឱ្យប្លែកស្ទាំងដប់ប្រាំ",\n      "generated_tokens_including_padding": 150,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 171,\n      "reference": "នៅក្នុងទីធ្លាព្រះវិហារ មានរូបចម្លាក់ជាថ្មម៉ាបដែលគួរឱ្យចាប់អារម្មណ៍នៅលើផ្នូរខ្លះ",\n      "hypothesis": "នៅក្នុងទីជ្រលៀងព្រះវិហារ មានរូបចម្លាក់ជាថ្មម៉ាបដែលគួរឱ្យចាប់អារម្មណ៍នៅលើភ្នំខ្លះ",\n      "generated_tokens_including_padding": 150,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 172,\n      "reference": "ដោយសារតែសិស្សនិស្សិតគឺជាទស្សនិកជនដែលធ្វើការរិះគន់ច្រើនជាគេ អ្នកសរសេរប្លក់ចាប់ផ្តើមព្យយាមកែលម្អការសរសេរដើម្បីចៀសវាងមតិរិះគន់",\n      "hypothesis": "ដោយសារតែសិស្សនិស្សិតគឺជាទូរស័ព្ទជនដែលធ្វើការរិះគន់កាន់តែងគេ និងសរសេរប្លក់ ចាប់ផ្តើមព្យាយាមកែលម្អការសរសេរ ដើម្បីជៀសវាងមតិនិគន់",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 173,\n      "reference": "បន្ទាប់មក ឡាក់ខាស៊ីងបាននាំមុខក្នុងការច្រៀងចម្រៀងផាចានស៍",\n      "hypothesis": "បន្ទាប់មក ឡាក់កៅស៊ីងបានដំបូងក្នុងការច្រៀងចម្រៀង ផាចាន",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 174,\n      "reference": "សត្វខ្លះដូចជា ដំរី និង សត្វកវែង ព្យាយាមចូលមកជិតឡាន ដូច្នេះសម្ភារៈតាមបទដ្ឋាន គឺអំណោយផលក្នុងការមើលទេសភាពបានល្អ",\n      "hypothesis": "សត្វខ្លះដូចជា ដំរី និងសត្វកវែង ព្យាយាមចូលមកជន្លាន ដូច្នេះសម្ភារៈអត់ឧបត្ថម្ភ គឺអំណោយផលក្នុងការមើលទេសភាពបានល្អ",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 177,\n      "reference": "មិនមានព្រៃឈើធំ ៗ នៅក្នុងដែនដីកាណាន canaan ទេ ដូច្នេះ ឈើមានតម្លៃថ្លៃណាស់",\n      "hypothesis": "មិនមានព្រៃឈើធំៗនៅក្នុងដែនដីកាន់ណាទី ដូច្នេះឈើមានតម្លៃថ្លៃណាស់",\n      "generated_tokens_including_padding": 155,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 178,\n      "reference": "ផលិតផលសិប្បកម្មអាចត្រូវបានចាត់ទុកថាជាវត្ថុបុរាណ ទោះបីជាវាមានអាយុតិចកាលជាងទំនិញស្រដៀងគ្នាដែល ផលិតបានច្រើនក៏ដោយ",\n      "hypothesis": "ផលិតផលសិប្បកម្មអាចត្រូវបានចាត់ទុកថាជាវត្ថុបុរាណ ទោះបីជាវាមានអាយុតិចកាលជាងទំនិញស្រដៀងគ្នាដែលផលិតបានច្រើនក៏ដោយ",\n      "generated_tokens_including_padding": 129,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 180,\n      "reference": "ទ្រង់ទ្រាយទូទៅជាច្រើន ឧទាហរណ៍ ទ្រង់ទ្រាយនៃក្រុម aps មានចំនួនស្មើនឹង ឬប្រហាក់ប្រហែលទៅនឹងសមាមាត្រមួយនេះ",\n      "hypothesis": "ទ្រង់ទ្រាយទូទៅជាច្រើន និទានហោជទ្រង់ទ្រាយនៃក្រុម APS មានចំនួនស្មើនឹង ឬប្រហាក់ប្រហែលទៅនឹងសមាមាត្រមួយនេះ",\n      "generated_tokens_including_padding": 129,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 184,\n      "reference": "គាត់បាននិយាយថា គាត់បានផលិតកន្តឹងទ្វារ wifi",\n      "hypothesis": "គាត់បាននិយាយថា គាត់បានផលិតកណ្តឹងទ្វារ Wi-Fi",\n      "generated_tokens_including_padding": 129,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 190,\n      "reference": "នៅពេលអ្នកទូរសព្ទហៅនរណាម្នាក់ដែលនៅចម្ងាយរាប់ពាន់គីឡូម៉ែត្រ អ្នក គឺកំពុងប្រើប្រាស់ផ្កាយរណបហើយ",\n      "hypothesis": "នៅពេលអ្នកទូរសព្ទហៅនរណាម្នាក់ដែលនៅចម្ងាយរាប់ពាន់ខ្លួនម៉ែត្រ អ្នកគឺកំពុងប្រើប្រាស់កាយរណបហើយ",\n      "generated_tokens_including_padding": 129,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 194,\n      "reference": "វាត្រូវបានគេគ្រោងនឹងកាត់បន្ថយនៅថ្ងៃអង្គារនេះទេ ប៉ុន្តែវាត្រូវបានរក្សាទុកសម្រេចពេលក្រោយបន្ទាប់ពីសេចក្តីសម្រេចរបស់តុលាការបន្ទាន់",\n      "hypothesis": "វាត្រូវបានគេគ្រោះនិងកាត់បន្ថយនៅថ្ងៃអង្គាណនេះទេ ប៉ុន្តែវាត្រូវបានរក្សាទុកសម្រេចពេលក្រោយបន្ទាប់ពីសេចក្តីសម្រេចរបស់តុលាការបន្ទាន់",\n      "generated_tokens_including_padding": 154,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 196,\n      "reference": "គួបផ្សំនឹងភាពមិនទាក់ទងគ្នា ពាក្យ timbuktu ត្រូវបានគេប្រើជាពាក្យប្រៀបធៀបសម្រាប់ដែនដីនៅឆ្ងាយនិងប្លែកថ្មី",\n      "hypothesis": "គួបផ្សំនឹងភាពមិនទាក់ទងគ្នា ពាក្យ ធីមបុកទូ ត្រូវបានគេប្រើជាពាក្យប្រៀបធៀបសម្រាប់ដែនដីនៅឆ្ងាយ និងប្លែកថ្មី",\n      "generated_tokens_including_padding": 154,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 197,\n      "reference": "វាក៏ពុំមានសិទ្ធិអំណាច ដើម្បីបដិសេធច្បាប់ពន្ធ និងពន្ធគយរវាងរដ្ឋនីមួយៗនោះដែរ",\n      "hypothesis": "វាក៏ពុំមានសុទ្ធអំណាច ដើម្បីបដិសណ្ឋានប្រពន្ធ និងពន្យល់យ៉ាវរវាងរដ្ឋនីមួយៗតែងតែ",\n      "generated_tokens_including_padding": 154,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    },\n    {\n      "validation_index": 198,\n      "reference": "សម្លការីអាច គោក ឬ រាវ អាស្រ័យលើបរិមាណអង្គធាតុរាវ ដាក់ទឹកតិចឬច្រើន",\n      "hypothesis": "សម្លការី អាចគោក ឬរាវ អាស្រ័យលើបរិមាណអង្គធាតុលាវ ដាក់ទឹកតិច ឬច្រើន",\n      "generated_tokens_including_padding": 154,\n      "has_eos": true,\n      "primary_has_eos": true,\n      "greedy_fallback": false\n    }\n  ],\n  "elapsed_seconds": 356.66038835100085,\n  "primary_result": {\n    "path": "results/diagnostic/qwen_base_bf16_beam3_validation_20261007.json",\n    "sha256": "e00678d7010efd69362024ba944fe725609e9436de0912435a58228f7fa5180e"\n  },\n  "primary_metrics": {\n    "evaluated_examples": 124,\n    "cer_percent": 14.976310929093392,\n    "wer_percent_icu_segmented": 30.985387785687525,\n    "cer_reference_characters": 12453,\n    "wer_reference_words": 2669,\n    "cer_errors": 1865,\n    "wer_errors": 827,\n    "icu_version": "77.1"\n  },\n  "metrics": {\n    "evaluated_examples": 124,\n    "cer_percent": 12.583313257849515,\n    "wer_percent_icu_segmented": 28.400149868864744,\n    "cer_reference_characters": 12453,\n    "wer_reference_words": 2669,\n    "cer_errors": 1567,\n    "wer_errors": 758,\n    "icu_version": "77.1"\n  },\n  "greedy_retry_count": 1\n}\n'}
for relative, source in PAYLOAD.items():
    destination = WORK/relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
os.environ.pop('KHMER_SCORE_PYTHON', None)
os.environ.pop('HF_HUB_OFFLINE', None)
os.environ.pop('HF_DATASETS_OFFLINE', None)
os.environ['HF_HOME'] = '/content/khmer_hf_cache'
os.environ['PYTHONUNBUFFERED'] = '1'
check = """
import sys, json, torch, icu
sys.path.insert(0, 'src')
from score_qwen_rows import score_rows
assert torch.cuda.is_available(), 'Select a GPU runtime'
assert icu.ICU_VERSION == '77.1', f'Unexpected ICU {icu.ICU_VERSION}'
with open('results/diagnostic/qwen_base_bf16_guarded_validation_20261007.json') as f:
    saved = json.load(f)
actual = score_rows(saved['examples'])
assert actual == saved['metrics'], (actual, saved['metrics'])
print('GPU:', torch.cuda.get_device_name(0))
print('Precision:', 'BF16' if torch.cuda.is_bf16_supported(including_emulation=False) else 'FP32')
print('Scoring regression passed:', actual)
"""
subprocess.run([PYTHON, '-c', check], cwd=WORK, check=True)


## Train and resume
Keep the run name unchanged to resume after a disconnect. If you change the recipe, choose a new run name. Only completed checkpoints are eligible for automatic resume. Each epoch is scored on all 124 validation clips. A completed or early-stopped run is not extended automatically.


In [ ]:
RUN_NAME = 'qwen_attention_r16_lr5e5_colab_v2_20261008'
OUT = Path('/content/drive/MyDrive/khmer_asr_runs')/RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
history_path = OUT/'history.json'
history = json.loads(history_path.read_text()) if history_path.exists() else []
done = (OUT/'finished.json').exists() or len(history) >= 4 or any(r['cer_percent'] < 20 and r['wer_percent_icu_segmented'] < 20 for r in history)
command = [PYTHON, 'src/train_qwen_candidate.py',
    '--manifest', 'results/diagnostic/qwen_train30_split_20261007.json',
    '--output-dir', str(OUT), '--cache-dir', '/content/khmer_qwen_audio',
    '--epochs', '4', '--patience', '2', '--lr', '5e-5', '--rank', '16',
    '--targets', 'q_proj,k_proj,v_proj,o_proj', '--grad-acc', '8',
    '--eval-beams', '3', '--eval-force-khmer', '--eval-batch-size', '1', '--dtype', 'auto']
candidates = []
for marker in OUT.glob('*/complete.json'):
    if (marker.parent/'training_state.pt').exists() and (marker.parent/'adapter_model.safetensors').exists():
        state = json.loads(marker.read_text())
        candidates.append(((state['step'], state['epoch']), marker.parent))
if candidates:
    latest = max(candidates, key=lambda item: item[0])[1]
    command += ['--resume', str(latest)]
    print('Resuming:', latest)
if done:
    print('Run already completed; showing saved results below.')
else:
    with (OUT/'training.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=WORK, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end=''); log.write(line); log.flush()
            code = process.wait()
            if code:
                raise RuntimeError(f'Training failed with exit code {code}; see {OUT}/training.log')
        except KeyboardInterrupt:
            process.terminate(); process.wait()
            raise


In [ ]:
history = json.loads((OUT/'history.json').read_text())
for row in history:
    print(f"Epoch {row['epoch']}: CER {row['cer_percent']:.2f}% | WER {row['wer_percent_icu_segmented']:.2f}%")
best = min(history, key=lambda row: row['wer_percent_icu_segmented'])
checkpoint = OUT/f"epoch-{best['epoch']}"
passed = best['cer_percent'] < 20 and best['wer_percent_icu_segmented'] < 20
print('Selected checkpoint:', checkpoint)
print('Validation target met:', passed)
print('Held-out test: not evaluated. Validation success is not a final test result.')
import zipfile
archive = OUT/'selected_adapter_and_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name in ['adapter_model.safetensors', 'adapter_config.json', 'validation_predictions.json']:
        bundle.write(checkpoint/name, name)
    for name in ['run_config.json', 'history.json', 'best_validation.json', 'training.log']:
        if (OUT/name).exists(): bundle.write(OUT/name, name)
    for path in (OUT/'source').glob('*.py'):
        bundle.write(path, 'source/'+path.name)
print('Saved export:', archive)
from google.colab import files
files.download(str(archive))
